# HLS + CCDC Forest Monitoring
Publication-ready notebook for comparing `L30`, `S30`, and combined `HLS` time series for forest disturbance monitoring with Google Earth Engine CCDC.

## What This Notebook Does
- quantifies the observation-density advantage of combined HLS relative to Landsat-only
- runs pixel-level CCDC fits and extracts breakpoint timing
- builds spatial disturbance maps for `L30`, `S30`, and `HLS`
- validates spatial CCDC results against Hansen Global Forest Change
- optionally scouts global or regional hotspot locations before site selection

## Prerequisites
- an authenticated Google Earth Engine Python environment
- Python packages used in this notebook: `earthengine-api`, `numpy`, `pandas`, `matplotlib`, `geopandas`, and `geemap`
- optional: precomputed GEE CCDC assets for `L30`, `S30`, and `HLS` if you want faster spatial runs

## Recommended Workflow
1. Run the setup/import cell first.
2. Optionally use the Hansen and hotspot-scanning utilities near the top of the notebook to choose a site.
3. Edit the **Site Configuration** cell only.
4. Run the notebook top-to-bottom for the core analysis.
5. Use the Hansen preview, validation, diagnostics, and report cells as the publication/QA layer.

## Main Outputs
- **Figure 1**: annual cloud-free observation frequency
- **Figure 2**: pixel time series, CCDC fits, and representative clear-sky snapshots
- **Figure 3**: spatial disturbance maps for `L30`, `S30`, and `HLS`
- **Figure 4**: Hansen-based validation metrics and comparison plots
- optional extras: animation frames/GIFs, hotspot scouting tables/maps, diagnostics, and a Markdown validation report

## Reproducibility Notes
- the notebook now uses repo-relative output paths where possible
- optional basemap files are discovered automatically when available, with a built-in fallback for global overview maps
- outputs are written under the notebook `figures/` directory, usually grouped by site

## Code Organization
- notebook orchestration and narrative stay in the notebook
- reusable functions live in `gee_hls/notebook_helpers.py`


In [ ]:
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import geopandas as gpd
import geemap
from datetime import datetime, timedelta
from collections import Counter
from pathlib import Path
import os
import sys
import warnings
warnings.filterwarnings('ignore')


def _find_project_root(start=None):
    """Locate the repository root so notebook paths stay portable."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'gee_hls').exists():
            return candidate
        if candidate.name == 'gee_hls':
            return candidate.parent
    return start


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = (
    PROJECT_ROOT / 'notebooks'
    if (PROJECT_ROOT / 'notebooks').exists()
    else Path.cwd().resolve()
)
FIGURES_DIR = NOTEBOOK_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

_BASEMAP_CANDIDATES = [
    NOTEBOOK_DIR / 'data' / 'ne_110m_admin_0_countries.shp',
    PROJECT_ROOT / 'data' / 'ne_110m_admin_0_countries.shp',
    Path.home() / 'Documents' / '110m_cultural' / 'ne_110m_admin_0_countries.shp',
]
WORLD_BASEMAP_PATH = next((str(path) for path in _BASEMAP_CANDIDATES if path.exists()), None)
HOTSPOT_TABLE_CSV = str(NOTEBOOK_DIR / 'top_100_forest_loss_hansen.csv')

# --- GEE authentication ---
# First run, authenticate if needed:
# ee.Authenticate()
GEE_PROJECT = os.environ.get('EE_PROJECT')
if GEE_PROJECT:
    ee.Initialize(project=GEE_PROJECT)
else:
    ee.Initialize()

print('Earth Engine initialized.')
print(f'Project root   : {PROJECT_ROOT}')
print(f'Notebook dir   : {NOTEBOOK_DIR}')
print(f'Figures dir    : {FIGURES_DIR}')
if WORLD_BASEMAP_PATH:
    print(f'World basemap  : {WORLD_BASEMAP_PATH}')
else:
    print('World basemap  : not found; global map cells will use built-in GeoPandas data when available.')
print('Helper module  : gee_hls/notebook_helpers.py')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HANSEN BASELINE-YEAR TEST — pseudo forest baseline from a chosen reference year
#
# Hansen provides treecover2000 as a fixed baseline. To approximate a later
# baseline year (for example 2015 to align more closely with HLS-era analysis),
# we keep pixels that:
#   1. met a tree-cover threshold in 2000, and
#   2. were not lost before the chosen reference year.
#
# This cell tests that logic over a small Amazon region and compares baseline
# forest area across several reference years.
# ═══════════════════════════════════════════════════════════════════════════════

HANSEN_TEST_REGION = ee.Geometry.Rectangle(
    [-61.5, -11.5, -59.5, -9.5], geodesic=False
)
HANSEN_TEST_LABEL = 'Southwestern Amazon test window'
HANSEN_TREECOVER_THRESHOLD = 30
HANSEN_REFERENCE_YEARS = [2000, 2010, 2015, 2020, 2024]
HANSEN_AREA_SCALE = 30

gfc = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
treecover2000 = gfc.select('treecover2000')
lossyear = gfc.select('lossyear')


def hansen_baseline_mask(reference_year, treecover_threshold=30, require_survive_through_year=False):
    """Approximate a Hansen forest baseline for a chosen reference year.

    If require_survive_through_year is False:
        keep pixels that were still forest at the start of reference_year.
        Example: reference_year=2015 removes losses from 2001-2014.

    If require_survive_through_year is True:
        keep pixels that survived through the end of reference_year.
        Example: reference_year=2015 also removes pixels lost in 2015.
    """
    base = treecover2000.gte(treecover_threshold)

    if reference_year <= 2000:
        return base.selfMask()

    cutoff = reference_year - 2000
    if require_survive_through_year:
        keep = lossyear.eq(0).Or(lossyear.gt(cutoff))
    else:
        keep = lossyear.eq(0).Or(lossyear.gte(cutoff))

    return base.And(keep).selfMask()


def masked_area_ha(mask_img, region, scale=30):
    area_img = ee.Image.pixelArea().divide(1e4).updateMask(mask_img)
    stats = area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e13,
        tileScale=4,
    )
    return ee.Number(stats.get('area', 0))


def annual_loss_mask(start_year, end_year):
    return (
        lossyear.gte(start_year - 2000)
        .And(lossyear.lte(end_year - 2000))
        .selfMask()
    )


baseline_rows = []
baseline_masks = {}
for year in HANSEN_REFERENCE_YEARS:
    mask = hansen_baseline_mask(year, HANSEN_TREECOVER_THRESHOLD)
    baseline_masks[year] = mask
    area_ha = masked_area_ha(mask.rename('area'), HANSEN_TEST_REGION, scale=HANSEN_AREA_SCALE)
    baseline_rows.append({
        'reference_year': year,
        'forest_area_ha': area_ha.getInfo(),
    })

df_hansen_baseline = pd.DataFrame(baseline_rows)
df_hansen_baseline['forest_area_kha'] = df_hansen_baseline['forest_area_ha'] / 1000.0
df_hansen_baseline['forest_area_pct_of_2000'] = (
    100 * df_hansen_baseline['forest_area_ha'] / max(df_hansen_baseline['forest_area_ha'].iloc[0], 1)
)

print(f'Hansen baseline-year test region: {HANSEN_TEST_LABEL}')
print('Pseudo-baseline rule: treecover2000 >= threshold AND no Hansen loss before reference year.')
print('\nForest area summary (Amazon test region):')
print(
    df_hansen_baseline.assign(
        forest_area_ha=lambda d: d['forest_area_ha'].round(0).astype(int),
        forest_area_kha=lambda d: d['forest_area_kha'].round(2),
        forest_area_pct_of_2000=lambda d: d['forest_area_pct_of_2000'].round(2),
    ).to_string(index=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    df_hansen_baseline['reference_year'].astype(str),
    df_hansen_baseline['forest_area_kha'],
    color='#2E7D32'
)
axes[0].set_title('Pseudo-baseline forest area by reference year', fontweight='bold')
axes[0].set_ylabel('Forest area (thousand ha)')
axes[0].set_xlabel('Reference year')

axes[1].plot(
    df_hansen_baseline['reference_year'],
    df_hansen_baseline['forest_area_pct_of_2000'],
    marker='o',
    color='#1565C0',
    linewidth=2
)
axes[1].set_title('Remaining baseline forest relative to year 2000', fontweight='bold')
axes[1].set_ylabel('Forest area (% of 2000 baseline)')
axes[1].set_xlabel('Reference year')
axes[1].set_ylim(0, 105)
axes[1].grid(True, linestyle='--', alpha=0.25)

fig.suptitle('Testing Hansen pseudo-baseline years over a small Amazon region', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
baseline_test_path = FIGURES_DIR / 'hansen_baseline_year_test_amazon.png'
fig.savefig(baseline_test_path, dpi=150, bbox_inches='tight')
print(f'\nSaved summary figure -> {baseline_test_path}')
plt.show()

# Optional visual preview map in the notebook.
loss_2015_2024 = annual_loss_mask(2015, 2024)
loss_2020_2024 = annual_loss_mask(2020, 2024)
preview_map = geemap.Map(center=[-10.5, -60.5], zoom=7)
preview_map.add_basemap('SATELLITE')
preview_map.addLayer(HANSEN_TEST_REGION, {}, 'Amazon test region', False)
preview_map.addLayer(
    baseline_masks[2000],
    {'palette': ['2E7D32']},
    f'Baseline 2000 (treecover>={HANSEN_TREECOVER_THRESHOLD}%)',
    False,
)
preview_map.addLayer(
    baseline_masks[2015],
    {'palette': ['66BB6A']},
    f'Pseudo-baseline 2015',
    True,
)
preview_map.addLayer(
    baseline_masks[2020],
    {'palette': ['A5D6A7']},
    f'Pseudo-baseline 2020',
    False,
)
preview_map.addLayer(
    loss_2015_2024.updateMask(loss_2015_2024),
    {'palette': ['FB8C00']},
    'Hansen loss 2015-2024',
    False,
)
preview_map.addLayer(
    loss_2020_2024.updateMask(loss_2020_2024),
    {'palette': ['E53935']},
    'Hansen loss 2020-2024',
    True,
)
preview_map


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HANSEN VALIDATION HELPER — reusable forest and loss layers for 2015-2024
#
# Use these layers to compare spatial CCDC disturbance maps against Hansen:
# - forest_mask_2000: baseline forest eligibility mask
# - loss_2015_2024: binary forest-loss mask for 2015-2024
# - loss_year_2015_2024: calendar-year loss image for timing comparison
# ═══════════════════════════════════════════════════════════════════════════════

HANSEN_VALIDATION_TREECOVER_THRESHOLD = 30
HANSEN_VALIDATION_START_YEAR = 2015
HANSEN_VALIDATION_END_YEAR = 2024

gfc = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
treecover2000 = gfc.select('treecover2000')
lossyear = gfc.select('lossyear')

forest_mask_2000 = treecover2000.gte(HANSEN_VALIDATION_TREECOVER_THRESHOLD).selfMask()

loss_2015_2024 = (
    forest_mask_2000
    .And(lossyear.gte(HANSEN_VALIDATION_START_YEAR - 2000))
    .And(lossyear.lte(HANSEN_VALIDATION_END_YEAR - 2000))
    .selfMask()
    .rename('loss_2015_2024')
)

loss_year_2015_2024 = (
    lossyear
    .updateMask(loss_2015_2024)
    .add(2000)
    .rename('loss_year_2015_2024')
)

print('Hansen validation layers ready:')
print(f'  forest_mask_2000        -> treecover2000 >= {HANSEN_VALIDATION_TREECOVER_THRESHOLD}%')
print(f'  loss_2015_2024          -> Hansen loss between {HANSEN_VALIDATION_START_YEAR} and {HANSEN_VALIDATION_END_YEAR}')
print('  loss_year_2015_2024     -> calendar-year version of lossyear for masked pixels')

# Quick Amazon preview for sanity check
validation_preview = geemap.Map(center=[-10.5, -60.5], zoom=7)
validation_preview.add_basemap('SATELLITE')
validation_preview.addLayer(
    forest_mask_2000,
    {'palette': ['2E7D32']},
    'forest_mask_2000',
    False,
)
validation_preview.addLayer(
    loss_2015_2024,
    {'palette': ['FB8C00']},
    'loss_2015_2024',
    True,
)
validation_preview.addLayer(
    loss_year_2015_2024,
    {'min': HANSEN_VALIDATION_START_YEAR, 'max': HANSEN_VALIDATION_END_YEAR,
     'palette': ['FFF7BC', 'FEC44F', 'FE9929', 'EC7014', 'CC4C02']},
    'loss_year_2015_2024',
    False,
)
validation_preview


---
## Optional Scouting Utilities
The next cells are optional helpers for publication-quality site selection and validation planning.

They let you:
- test Hansen baseline-year logic
- create reusable Hansen forest/loss masks
- identify global forest-loss hotspots
- map hotspot locations and compare expected HLS observation gains
- survey candidate sites before committing to a detailed CCDC run

If you already know your study site, you can skip ahead to **Site Configuration**.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL HOTSPOT SCOUT — rank recent forest-loss hotspots from Hansen GFC
#
# This cell scans a coarse global grid for places with both substantial
# baseline forest cover and strong recent tree-cover loss. The output is a
# shortlist of candidate hotspot coordinates that you can inspect manually
# before copying one into SITE_CONFIG.
#
# Notes:
# - Uses Hansen Global Forest Change v1.12 (2000-2024).
# - Uses a direct Hansen loss window:
#       forest_mask_2000 AND (lossyear between HOTSPOT_START_YEAR and HOTSPOT_END_YEAR)
# - Returns coarse grid-cell centroids, not exact disturbed pixels.
# - For HLS demos, set HOTSPOT_SCOPE='tropics' to emphasize deforestation
#   frontiers and avoid boreal fire hotspots.
# ═══════════════════════════════════════════════════════════════════════════════

HOTSPOT_SCOPE = 'global'       # 'global' or 'tropics'
HOTSPOT_START_YEAR = 2015
HOTSPOT_END_YEAR = 2024
HOTSPOT_TREECOVER_MIN = 30     # forest threshold from treecover2000 (%)
HOTSPOT_GRID_DEG = 4          # smaller cells = more local detail, slower
HOTSPOT_MIN_FOREST_HA = 100_000
HOTSPOT_MIN_LOSS_HA = 5_000
HOTSPOT_TOP_N = 100
HOTSPOT_SCALE = 300            # aggregation scale in meters

if HOTSPOT_SCOPE == 'tropics':
    HOTSPOT_BOUNDS = ee.Geometry.Rectangle(
        [-180, -30, 180, 30], geodesic=False
    )
else:
    HOTSPOT_BOUNDS = ee.Geometry.Rectangle(
        [-180, -60, 180, 75], geodesic=False
    )


def build_hotspot_grid(bounds, step_deg):
    """Create a coarse lat/lon grid as an ee.FeatureCollection server-side.

    This avoids client-side getInfo() calls and prevents large payload errors
    when scanning global domains.
    """
    bounds = ee.Geometry(bounds).transform('EPSG:4326', 1)
    coords = ee.List(bounds.coordinates().get(0))

    xs = coords.map(lambda pt: ee.Number(ee.List(pt).get(0)))
    ys = coords.map(lambda pt: ee.Number(ee.List(pt).get(1)))

    x_min = ee.Number(xs.reduce(ee.Reducer.min()))
    x_max = ee.Number(xs.reduce(ee.Reducer.max()))
    y_min = ee.Number(ys.reduce(ee.Reducer.min()))
    y_max = ee.Number(ys.reduce(ee.Reducer.max()))
    step = ee.Number(step_deg)

    lon_starts = ee.List.sequence(x_min, x_max.subtract(1e-9), step)
    lat_starts = ee.List.sequence(y_min, y_max.subtract(1e-9), step)

    def make_row(lat):
        lat = ee.Number(lat)
        next_lat = lat.add(step).min(y_max)

        def make_cell(lon):
            lon = ee.Number(lon)
            next_lon = lon.add(step).min(x_max)
            geom = ee.Geometry.Rectangle(
                [lon, lat, next_lon, next_lat],
                proj='EPSG:4326',
                geodesic=False
            )
            return ee.Feature(
                geom,
                {
                    'cell_id': ee.String(lat.format('%.1f')).cat('_').cat(lon.format('%.1f')),
                    'lon_min': lon,
                    'lon_max': next_lon,
                    'lat_min': lat,
                    'lat_max': next_lat,
                }
            )

        return lon_starts.map(make_cell)

    return ee.FeatureCollection(lat_starts.map(make_row).flatten())


gfc = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
treecover2000 = gfc.select('treecover2000')
lossyear = gfc.select('lossyear')

forest_mask_2000 = treecover2000.gte(HOTSPOT_TREECOVER_MIN).selfMask()
loss_window_mask = (
    forest_mask_2000
    .And(lossyear.gte(HOTSPOT_START_YEAR - 2000))
    .And(lossyear.lte(HOTSPOT_END_YEAR - 2000))
    .selfMask()
)
loss_year_window = (
    lossyear.updateMask(loss_window_mask)
    .add(2000)
    .rename('loss_year_window')
)

forest_area_ha = (
    ee.Image.pixelArea()
    .divide(1e4)
    .updateMask(forest_mask_2000)
    .rename('forest_ha')
)
loss_area_ha = (
    ee.Image.pixelArea()
    .divide(1e4)
    .updateMask(loss_window_mask)
    .rename('loss_ha')
)
metrics_img = forest_area_ha.addBands(loss_area_ha)


def summarize_hotspot_cell(feature):
    stats = metrics_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=feature.geometry(),
        scale=HOTSPOT_SCALE,
        bestEffort=True,
        maxPixels=1e13,
        tileScale=4,
    )

    forest_ha = ee.Number(stats.get('forest_ha', 0))
    loss_ha = ee.Number(stats.get('loss_ha', 0))
    loss_frac = loss_ha.divide(forest_ha.max(1))
    loss_pct = loss_frac.multiply(100)
    hotspot_score = loss_ha.multiply(loss_frac.sqrt())
    center = feature.geometry().centroid(1000).coordinates()

    return feature.set({
        'forest_ha': forest_ha,
        'loss_ha': loss_ha,
        'loss_pct': loss_pct,
        'hotspot_score': hotspot_score,
        'center_lon': center.get(0),
        'center_lat': center.get(1),
    })


print(
    f"Scanning Hansen hotspots ({HOTSPOT_START_YEAR}-{HOTSPOT_END_YEAR}) "
    f"at {HOTSPOT_GRID_DEG}° grid ..."
)
hotspot_grid = build_hotspot_grid(HOTSPOT_BOUNDS, HOTSPOT_GRID_DEG)
hotspot_fc = (
    hotspot_grid
    .map(summarize_hotspot_cell)
    .filter(ee.Filter.gte('forest_ha', HOTSPOT_MIN_FOREST_HA))
    .filter(ee.Filter.gte('loss_ha', HOTSPOT_MIN_LOSS_HA))
    .sort('hotspot_score', False)
    .limit(HOTSPOT_TOP_N)
)

hotspot_info = hotspot_fc.getInfo()['features']

if not hotspot_info:
    print('No hotspot cells matched the current filters.')
    print('Try lowering HOTSPOT_MIN_FOREST_HA / HOTSPOT_MIN_LOSS_HA or widening the scope.')
else:
    hotspot_rows = []
    for rank, feat in enumerate(hotspot_info, start=1):
        props = feat['properties']
        hotspot_rows.append({
            'rank': rank,
            'cell_id': props['cell_id'],
            'lon': props['center_lon'],
            'lat': props['center_lat'],
            'forest_ha': props['forest_ha'],
            'loss_ha': props['loss_ha'],
            'loss_pct': props['loss_pct'],
        })

    df_hotspots = pd.DataFrame(hotspot_rows)
    df_hotspots['forest_kha'] = df_hotspots['forest_ha'] / 1_000.0
    df_hotspots['loss_kha'] = df_hotspots['loss_ha'] / 1_000.0

    display_cols = ['rank', 'lon', 'lat', 'forest_ha', 'loss_ha', 'loss_pct', 'cell_id']
    print('\nTop hotspot cells:')
    print(
        df_hotspots[display_cols]
        .assign(
            lon=lambda d: d['lon'].round(3),
            lat=lambda d: d['lat'].round(3),
            forest_ha=lambda d: d['forest_ha'].round(0).astype(int),
            loss_ha=lambda d: d['loss_ha'].round(0).astype(int),
            loss_pct=lambda d: d['loss_pct'].round(2),
        )
        .to_string(index=False)
    )

    print('\nSuggested next step:')
    print('Inspect one of the top hotspot centroids in GEE or a basemap, then copy a nearby disturbed')
    print('pixel into SITE_CONFIG for the detailed HLS + CCDC workflow.')
    print('\nTop coordinates to inspect:')
    for row in df_hotspots.head(5).itertuples():
        print(
            f"  {row.rank}. lon={row.lon:.4f}, lat={row.lat:.4f}  "
            f"| loss={row.loss_ha:,.0f} ha ({row.loss_pct:.1f}% of baseline forest)"
        )


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL HOTSPOT LOCATION MAP — matplotlib view of the top hotspot centroids
#
# Requires df_hotspots from the hotspot-scout cell above.
# Uses a local Natural Earth country shapefile as the baselayer.
# ═══════════════════════════════════════════════════════════════════════════════

from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon
from gee_hls.notebook_helpers import read_polygon_shapefile

HOTSPOT_BASEMAP = WORLD_BASEMAP_PATH

if 'df_hotspots' not in globals() or df_hotspots.empty:
    print('df_hotspots not found — run the GLOBAL HOTSPOT SCOUT cell first.')
else:
    hotspot_polygons = []
    if HOTSPOT_BASEMAP and Path(HOTSPOT_BASEMAP).exists():
        hotspot_polygons = read_polygon_shapefile(HOTSPOT_BASEMAP)
    else:
        print('Optional shapefile basemap not found; using a built-in GeoPandas world layer when available.')

    fig, ax = plt.subplots(figsize=(16, 8), constrained_layout=True)
    ax.set_facecolor('#DCEFF8')

    if hotspot_polygons:
        country_patches = [Polygon(poly, closed=True) for poly in hotspot_polygons]
        country_collection = PatchCollection(
            country_patches,
            facecolor='#F7F4EA',
            edgecolor='#9A9A9A',
            linewidth=0.35,
            zorder=1,
        )
        ax.add_collection(country_collection)
    else:
        try:
            world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
            world.plot(ax=ax, color='#F7F4EA', edgecolor='#9A9A9A', linewidth=0.35, zorder=1)
        except Exception as exc:
            print(f'  Warning: built-in world basemap unavailable ({exc}). Plotting hotspot points without a basemap.')

    loss_min = float(df_hotspots['loss_pct'].min())
    loss_max = float(df_hotspots['loss_pct'].max())
    if np.isclose(loss_min, loss_max):
        loss_max = loss_min + 1.0

    size_scale = np.clip(df_hotspots['loss_pct'].to_numpy(dtype=float), 0, None)
    size_scale = 40 + 240 * (size_scale / max(loss_max, 1e-6)) ** 0.7

    scatter = ax.scatter(
        df_hotspots['lon'],
        df_hotspots['lat'],
        c=df_hotspots['loss_pct'],
        s=size_scale,
        cmap='YlOrRd',
        vmin=loss_min,
        vmax=loss_max,
        edgecolor='black',
        linewidth=0.45,
        alpha=0.9,
        zorder=3,
    )

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.82, pad=0.015)
    cbar.set_label('Forest loss (%) within hotspot cell', fontsize=12)

    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(
        f'Top Hansen Forest-Loss Hotspots ({HOTSPOT_START_YEAR}-{HOTSPOT_END_YEAR})',
        fontsize=15,
        fontweight='bold',
    )
    ax.grid(False)

    out_path = FIGURES_DIR / 'global_hotspot_locations_matplotlib.png'
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    print(f'Saved hotspot location map -> {out_path}')
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TOP 100 HOTSPOT OBSERVATION COUNTS — L30 vs S30 vs HLS
#
# Reuses the Hansen hotspot ranking from the previous cell, takes the top 100
# hotspot centroids, and counts cloud-free observations at each location.
# Run once; results are stored in df_top100_obs for the map cell below.
# ═══════════════════════════════════════════════════════════════════════════════

TOP100_N = 100

print(
    f'Building top {TOP100_N} hotspot centroids using Hansen hotspot_score '
    f'({HOTSPOT_START_YEAR}-{HOTSPOT_END_YEAR}) ...'
)

hotspot_fc_top100 = (
    build_hotspot_grid(HOTSPOT_BOUNDS, HOTSPOT_GRID_DEG)
    .map(summarize_hotspot_cell)
    .filter(ee.Filter.gte('forest_ha', HOTSPOT_MIN_FOREST_HA))
    .filter(ee.Filter.gte('loss_ha', HOTSPOT_MIN_LOSS_HA))
    .sort('hotspot_score', False)
    .limit(TOP100_N)
)

hotspot_top100_info = hotspot_fc_top100.getInfo()['features']

if not hotspot_top100_info:
    print('No hotspot locations matched the current filters.')
    print('Try lowering HOTSPOT_MIN_FOREST_HA / HOTSPOT_MIN_LOSS_HA or changing HOTSPOT_SCOPE.')
else:
    top100_rows = []
    print(f'Counting cloud-free observations at {len(hotspot_top100_info)} hotspot centroids ...')

    for idx, feat in enumerate(hotspot_top100_info, start=1):
        props = feat['properties']
        lon = props['center_lon']
        lat = props['center_lat']
        pt = ee.Geometry.Point([lon, lat])
        counts = count_clear_obs(pt, SURVEY_START, SURVEY_END)

        top100_rows.append({
            'original_hotspot_rank': idx,
            'cell_id': props['cell_id'],
            'lon': lon,
            'lat': lat,
            'loss_pct': props['loss_pct'],
            'l30_clear': counts['l30_clear'],
            's30_clear': counts['s30_clear'],
            'hls_clear': counts['hls_clear'],
        })

        if idx % 10 == 0 or idx == len(hotspot_top100_info):
            print(f'  Processed {idx}/{len(hotspot_top100_info)} locations')

    df_top100_obs = pd.DataFrame(top100_rows)

    # Combined rank favors locations with both strong HLS-vs-L30 observation gain
    # and high forest-loss intensity.
    df_top100_obs['gain_ratio'] = (
        df_top100_obs['hls_clear'] / df_top100_obs['l30_clear'].clip(lower=1)
    )
    gain_max = max(df_top100_obs['gain_ratio'].max(), 1)
    loss_max = max(df_top100_obs['loss_pct'].max(), 1)
    df_top100_obs['rank_score'] = (
        (df_top100_obs['gain_ratio'] / gain_max)
        * (df_top100_obs['loss_pct'] / loss_max)
    )
    df_top100_obs = (
        df_top100_obs
        .sort_values(
            ['rank_score', 'gain_ratio', 'loss_pct', 'hls_clear'],
            ascending=[False, False, False, False]
        )
        .reset_index(drop=True)
    )
    df_top100_obs['rank'] = np.arange(1, len(df_top100_obs) + 1)

    top100_csv_path = HOTSPOT_TABLE_CSV
    df_top100_obs[[
        'rank', 'original_hotspot_rank', 'cell_id', 'lon', 'lat',
        'loss_pct', 'gain_ratio', 'rank_score',
        'l30_clear', 's30_clear', 'hls_clear'
    ]].to_csv(top100_csv_path, index=False)
    print(f'\nExported ranked hotspot table -> {top100_csv_path}')

    print(f'\nDone. df_top100_obs has {len(df_top100_obs)} rows.')
    print("\nRank now reflects the combined priority of gain_ratio and loss_pct.")
    print('\nTop 10 by combined rank:')
    print(
        df_top100_obs.head(10)[[
            'rank', 'original_hotspot_rank', 'lon', 'lat',
            'gain_ratio', 'loss_pct', 'rank_score',
            'l30_clear', 's30_clear', 'hls_clear'
        ]]
        .assign(
            lon=lambda d: d['lon'].round(3),
            lat=lambda d: d['lat'].round(3),
            gain_ratio=lambda d: d['gain_ratio'].round(2),
            loss_pct=lambda d: d['loss_pct'].round(2),
            rank_score=lambda d: d['rank_score'].round(3),
        )
        .to_string(index=False)
    )


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TOP 100 HOTSPOT OBSERVATION MAPS — four-panel global map
#
# Requires df_top100_obs from the cell above.
# ═══════════════════════════════════════════════════════════════════════════════

TOP100_BASEMAP = WORLD_BASEMAP_PATH

if TOP100_BASEMAP and Path(TOP100_BASEMAP).exists():
    try:
        world_obs = gpd.read_file(TOP100_BASEMAP)
    except Exception as exc:
        print(f'Optional local basemap unavailable ({exc}), falling back to built-in ...')
        try:
            world_obs = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
        except Exception as exc2:
            print(f'Built-in basemap also unavailable: {exc2}')
            world_obs = None
else:
    print('Optional local basemap not found; falling back to built-in GeoPandas world layer ...')
    try:
        world_obs = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    except Exception as exc:
        print(f'Built-in basemap also unavailable: {exc}')
        world_obs = None

obs_cols = ['l30_clear', 's30_clear', 'hls_clear']
obs_max  = max(df_top100_obs[obs_cols].to_numpy().max(), 1)
loss_max = max(df_top100_obs['loss_pct'].max(), 1)

# HLS / L30 gain ratio: how many times more observations HLS gives vs L30 alone
df_top100_obs['gain_ratio'] = (
    df_top100_obs['hls_clear'] / df_top100_obs['l30_clear'].replace(0, float('nan'))
).fillna(1.0)
gain_max = max(df_top100_obs['gain_ratio'].max(), 1)

# size_max is shared (obs_max) so dot sizes are comparable across L30 / S30 rows.
panel_specs = [
    ('l30_clear',  'L30 Only (cloud-free obs)',  'Reds',   0, obs_max,  'Cloud-free obs',  'l30_clear',   obs_max),
    ('s30_clear',  'S30 Only (cloud-free obs)',  'Reds',   0, obs_max,  'Cloud-free obs',  's30_clear',   obs_max),
    ('gain_ratio', 'HLS / L30 Obs Gain Ratio',   'Blues', 1, gain_max, 'HLS obs / L30 obs', 'gain_ratio',  gain_max),
    ('loss_pct',  'Forest Loss (%)', 'Greens', 0, loss_max, 'Forest loss (%)', 'loss_pct',  loss_max),
]

fig, axes = plt.subplots(4, 1, figsize=(18, 20), constrained_layout=True)

for ax, (col, title, cmap, vmin, vmax, cbar_label, size_col, size_max) in zip(axes, panel_specs):
    ax.set_facecolor('#EAF4FB')
    if world_obs is not None:
        world_obs.plot(ax=ax, color='#F5F3EB', edgecolor='#9E9E9E', linewidth=0.4)

    scatter = ax.scatter(
        df_top100_obs['lon'],
        df_top100_obs['lat'],
        c=df_top100_obs[col],
        s=15 + 280 * (df_top100_obs[size_col] / size_max) ** 0.5,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        edgecolor='black',
        linewidth=0.35,
        alpha=0.85,
        zorder=3,
    )

    cbar = fig.colorbar(scatter, ax=ax, shrink=0.8, pad=0.01, aspect=25)
    cbar.set_label(cbar_label, fontsize=16)
    if col == 'gain_ratio':
        cbar.ax.text(0.5, 1.05, 'S30 fills gaps', transform=cbar.ax.transAxes,
                     ha='center', va='bottom', fontsize=16, color='#1B5E20', fontstyle='italic')
        cbar.ax.text(0.5, -0.05, 'L30 sufficient', transform=cbar.ax.transAxes,
                     ha='center', va='top', fontsize=16, color='#B71C1C', fontstyle='italic')

    ax.set_title(title, fontweight='bold')
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 80)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(False)

fig.suptitle(
    f'Top {len(df_top100_obs)} hotspot centroids — cloud-free observations & forest loss by sensor',
    fontsize=15,
    fontweight='bold',
)

top100_map_path = FIGURES_DIR / 'top100_hotspot_observation_maps.png'
fig.savefig(top100_map_path, dpi=150, bbox_inches='tight')
print(f'Map saved -> {top100_map_path}')
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SITE SURVEY — Count cloud-free L30 / S30 / HLS observations at candidate sites
#
# Run this cell BEFORE choosing a SITE_CONFIG to empirically find the site with
# the best L30-vs-S30 contrast and enough total observations for CCDC.
#
# Rule of thumb: CCDC needs ~6-8 observations per segment per band. For a
# 2016-2024 study period with 1-3 breaks, aim for 80+ HLS observations minimum.
# ═══════════════════════════════════════════════════════════════════════════════

CANDIDATE_SITES = {
    # ── SE Asia (seasonal tropics) ────────────────────────────────────────────
    'DakLak_Vietnam':        {'lon': 108.05,   'lat': 12.70,
                              'note': 'Rubber/coffee expansion, dry Nov-Apr'},
    'KeoSeima_Cambodia':     {'lon': 106.8421, 'lat': 12.3356,
                              'note': 'Border logging/ag expansion, dry Nov-Apr'},
    'PhnomPrich_Cambodia':   {'lon': 106.8656, 'lat': 12.7583,
                              'note': 'Logging corridor near Vietnam border, dry Nov-Apr'},
    'ChiangRai_Thailand':    {'lon': 100.20,   'lat': 19.50,
                              'note': 'Slash-and-burn, dry Nov-Mar'},
    'ShanState_Myanmar':     {'lon': 97.50,    'lat': 21.00,
                              'note': 'Ag/mining deforestation, dry Nov-Mar'},

    # ── Latin America (seasonal tropics) ──────────────────────────────────────
    'Rondonia_Brazil':       {'lon': -63.00,   'lat': -9.50,
                              'note': 'Fishbone deforestation, dry May-Sep'},
    'Peten_Guatemala':       {'lon': -90.00,   'lat': 17.20,
                              'note': 'Maya Biosphere Reserve, dry Feb-May'},

    # ── North America (temperate / Mediterranean) ───────────────────────────────
    'Angeles_California':    {'lon': -118.5601, 'lat': 34.0948,
                              'note': 'Wildfire, clear-cut harvest, post-fire regrowth; dry Jun-Oct'},

    # ── Top-5 global hotspots (Hansen GFC + HLS survey) ──────────────────────
    'Guinean_forests_of_West_Africa_H01':
                             {'lon': -12.5000, 'lat': 7.4987,
                              'note': 'Rank-1 hotspot: Guinean forests of West Africa; L30=9 S30=65 HLS=74 loss=22.1%'},
    'Southeast_Australian_forests_H02':
                             {'lon': 147.5000, 'lat': -37.4820,
                              'note': 'Rank-2 hotspot: Southeast Australian forests; L30=105 S30=407 HLS=512 loss=26.8%'},
    'Madagascar_dry_forests_H03':
                             {'lon': 47.5000, 'lat': -17.4963,
                              'note': 'Rank-3 hotspot: Madagascar dry forests; L30=122 S30=466 HLS=588 loss=19.3%'},
    'Western_Canadian_boreal_forest_H04':
                             {'lon': -117.5000, 'lat': 62.4385,
                              'note': 'Rank-4 hotspot: Western Canadian boreal forest; L30=74 S30=225 HLS=299 loss=21.4%'},
    'Central_European_mixed_forests_H05':
                             {'lon': 7.5000, 'lat': 52.4625,
                              'note': 'Rank-5 hotspot: Central European mixed forests; L30=157 S30=376 HLS=533 loss=20.9%'},

    # ── Baseline (current notebook default) ───────────────────────────────────
    'MatoGrosso_Brazil':     {'lon': -54.84,   'lat': -8.50,
                              'note': 'Current default site'},
}



from gee_hls.notebook_helpers import count_clear_obs

SURVEY_START = '2016-01-01'
SURVEY_END   = '2024-12-31'


def count_clear_obs(point, start, end):
    """Count cloud-free L30, S30, and combined HLS images at a point."""
    l30_all = (ee.ImageCollection('NASA/HLS/HLSL30/v002')
               .filterBounds(point)
               .filterDate(start, end))
    s30_all = (ee.ImageCollection('NASA/HLS/HLSS30/v002')
               .filterBounds(point)
               .filterDate(start, end))

    # Cloud-free: Fmask == 64 (clear land) or 128 (clear land, mod aerosol)
    def is_clear(img):
        fmask = img.select('Fmask')
        return img.set('clear', fmask.eq(64).Or(fmask.eq(128))
                       .reduceRegion(ee.Reducer.first(), point, 30)
                       .get('Fmask'))

    l30_tagged = l30_all.map(is_clear)
    s30_tagged = s30_all.map(is_clear)

    n_l30_total = l30_all.size().getInfo()
    n_s30_total = s30_all.size().getInfo()

    n_l30_clear = (l30_tagged.filter(ee.Filter.eq('clear', 1))
                   .size().getInfo())
    n_s30_clear = (s30_tagged.filter(ee.Filter.eq('clear', 1))
                   .size().getInfo())

    return {
        'l30_total': n_l30_total, 'l30_clear': n_l30_clear,
        's30_total': n_s30_total, 's30_clear': n_s30_clear,
        'hls_total': n_l30_total + n_s30_total,
        'hls_clear': n_l30_clear + n_s30_clear,
    }


# ── Run the survey ────────────────────────────────────────────────────────────
print(f'Surveying {len(CANDIDATE_SITES)} candidate sites ({SURVEY_START} to {SURVEY_END})...')
print(f'{"":-<90}')

survey_results = {}
for name, info in CANDIDATE_SITES.items():
    pt = ee.Geometry.Point([info['lon'], info['lat']])
    try:
        counts = count_clear_obs(pt, SURVEY_START, SURVEY_END)
        survey_results[name] = counts
        ratio = (counts['hls_clear'] / counts['l30_clear']
                 if counts['l30_clear'] > 0 else float('inf'))
        print(f"  {name:<25s}  L30: {counts['l30_clear']:>4d}  "
              f"S30: {counts['s30_clear']:>4d}  "
              f"HLS: {counts['hls_clear']:>4d}  "
              f"ratio: {ratio:.1f}x  "
              f"({info['note']})")
    except Exception as e:
        print(f"  {name:<25s}  ERROR: {e}")

print(f'{"":-<90}')

# ── Plot comparison ───────────────────────────────────────────────────────────
if survey_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    sites = list(survey_results.keys())
    l30_counts = [survey_results[s]['l30_clear'] for s in sites]
    s30_counts = [survey_results[s]['s30_clear'] for s in sites]
    hls_counts = [survey_results[s]['hls_clear'] for s in sites]

    x = np.arange(len(sites))
    w = 0.25

    # Bar chart: absolute counts
    ax1.bar(x - w, l30_counts, w, label='L30 (Landsat)', color='#1565C0')
    ax1.bar(x,     s30_counts, w, label='S30 (Sentinel-2)', color='#E53935')
    ax1.bar(x + w, hls_counts, w, label='HLS Combined', color='#2E7D32')
    ax1.set_xticks(x)
    ax1.set_xticklabels([s.replace('_', '\n') for s in sites],
                         fontsize=9, rotation=0)
    ax1.set_ylabel('Cloud-free observations (2016-2024)')
    ax1.set_title('Cloud-Free Observation Counts by Site', fontweight='bold')
    ax1.legend()
    ax1.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='~CCDC minimum')
    ax1.text(len(sites)-0.5, 82, 'CCDC minimum ~80', fontsize=8, color='gray',
             ha='right')

    # Bar chart: HLS/L30 ratio
    ratios = [h / l if l > 0 else 0 for h, l in zip(hls_counts, l30_counts)]
    colors = ['#2E7D32' if r >= 2.0 else '#FB8C00' if r >= 1.5 else '#E53935'
              for r in ratios]
    ax2.bar(x, ratios, 0.5, color=colors)
    ax2.set_xticks(x)
    ax2.set_xticklabels([s.replace('_', '\n') for s in sites],
                         fontsize=9, rotation=0)
    ax2.set_ylabel('HLS / L30 ratio')
    ax2.set_title('HLS Observation Gain Over Landsat Alone', fontweight='bold')
    ax2.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5)
    ax2.text(len(sites)-0.5, 2.05, '2× gain', fontsize=8, color='gray', ha='right')
    for i, r in enumerate(ratios):
        ax2.text(i, r + 0.05, f'{r:.1f}×', ha='center', fontsize=10, fontweight='bold')

    fig.suptitle('Candidate Site Survey — L30 vs S30 vs HLS',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    site_survey_path = FIGURES_DIR / 'site_survey.png'
    fig.savefig(site_survey_path, dpi=150, bbox_inches='tight')
    print(f'\nSurvey plot saved -> {site_survey_path}')
    plt.show()

    # ── Recommendation ────────────────────────────────────────────────────────
    print('\n' + '=' * 70)
    print('RECOMMENDATION')
    print('=' * 70)
    # Score: want high ratio AND enough total obs
    best = max(survey_results.items(),
               key=lambda kv: (kv[1]['hls_clear'] / max(kv[1]['l30_clear'], 1))
                              * min(kv[1]['hls_clear'] / 80, 1.0))  # penalty if < 80
    info = CANDIDATE_SITES[best[0]]
    c = best[1]
    r = c['hls_clear'] / max(c['l30_clear'], 1)
    print(f'  Best site: {best[0]}')
    print(f'  Coords:   lon={info["lon"]}, lat={info["lat"]}')
    print(f'  L30: {c["l30_clear"]}  S30: {c["s30_clear"]}  HLS: {c["hls_clear"]}  '
          f'(ratio: {r:.1f}×)')
    print(f'  {info["note"]}')
    print(f'\nCopy to SITE_CONFIG:')
    print(f"  'pixel_lon': {info['lon']},")
    print(f"  'pixel_lat': {info['lat']},")
    print('=' * 70)

# ── Auto-inject top-5 hotspot sites from df_top100_obs (if available) ─────────
try:
    _top5 = df_top100_obs.sort_values('rank').head(5)
    _added = []
    for _, row in _top5.iterrows():
        key = f"Hotspot_{int(row['rank']):02d}"
        CANDIDATE_SITES[key] = {
            'lon':  round(row['lon'], 4),
            'lat':  round(row['lat'], 4),
            'note': (f"Top-{int(row['rank'])} hotspot  "
                     f"L30={int(row['l30_clear'])}  "
                     f"S30={int(row['s30_clear'])}  "
                     f"HLS={int(row['hls_clear'])}  "
                     f"loss={row['loss_pct']:.1f}%"),
        }
        _added.append(key)
    print(f'\nAdded {len(_added)} hotspot sites to CANDIDATE_SITES: {_added}')
except NameError:
    print('\n(df_top100_obs not found — run the hotspot observation cell first to add hotspot sites.)')

---
## Site Configuration
Edit only the next code cell to run the full workflow for a new site.

Recommended edits:
- choose a `SITE_NAME` from `CANDIDATE_SITES`, or keep `SITE_NAME = 'custom'`
- if using `custom`, replace the longitude, latitude, and note
- set the year range and, if available, provide precomputed CCDC asset paths

By default, outputs are written to a site-specific folder inside the notebook `figures/` directory.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SITE SELECTION  ←  only edit this block
# Pick a name from CANDIDATE_SITES above, or enter custom coordinates below.
# ─────────────────────────────────────────────────────────────────────────────
SITE_NAME = 'custom'   # choose a key from CANDIDATE_SITES, or keep 'custom'

# Custom override — used only when SITE_NAME == 'custom'
CUSTOM = {
    'pixel_lon': -74.65,
    'pixel_lat': -1.75,
    'note': 'Replace with your study-site description',
}

# ── Time range ────────────────────────────────────────────────────────────────
START_YEAR = 2015
END_YEAR   = 2024

# ── Analysis settings (rarely need changing) ──────────────────────────────────
REGION_BUFFER_DEG = 0.55           # spatial CCDC map buffer (±deg, ~55 km)
PLOT_BANDS        = ['NDVI', 'NBR']
BREAKPOINT_BANDS  = ['GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'NBR']
MIN_OBSERVATIONS  = 8

# Pre-computed CCDC assets — set to None to compute on-the-fly
CCDC_ASSET_HLS = None   # e.g. 'users/trangthuyvohcmus/ccdc_hls_phnomprich'
CCDC_ASSET_S30 = None
CCDC_ASSET_L30 = None

OUTPUT_DIR = str(FIGURES_DIR / SITE_NAME)
# ─────────────────────────────────────────────────────────────────────────────

# ── Build SITE_CONFIG from the inputs above (do not edit below) ───────────────
if SITE_NAME == 'custom':
    _info = CUSTOM
else:
    if SITE_NAME not in CANDIDATE_SITES:
        raise ValueError(f"'{SITE_NAME}' not found in CANDIDATE_SITES. "
                         f"Available: {list(CANDIDATE_SITES.keys())}")
    _info = CANDIDATE_SITES[SITE_NAME]

SITE_CONFIG = {
    'site_name':         SITE_NAME,
    'description':       _info['note'],
    'pixel_lon':         _info['lon'] if SITE_NAME != 'custom' else _info['pixel_lon'],
    'pixel_lat':         _info['lat'] if SITE_NAME != 'custom' else _info['pixel_lat'],
    'region_buffer_deg': REGION_BUFFER_DEG,
    'start_year':        START_YEAR,
    'end_year':          END_YEAR,
    'plot_bands':        PLOT_BANDS,
    'breakpoint_bands':  BREAKPOINT_BANDS,
    'min_observations':  MIN_OBSERVATIONS,
    'ccdc_asset_hls':    CCDC_ASSET_HLS,
    'ccdc_asset_s30':    CCDC_ASSET_S30,
    'ccdc_asset_l30':    CCDC_ASSET_L30,
    'output_dir':        OUTPUT_DIR,
    'map_year_min':      START_YEAR,
    'map_year_max':      END_YEAR,
}

_lon = SITE_CONFIG['pixel_lon']
_lat = SITE_CONFIG['pixel_lat']
_buf = SITE_CONFIG['region_buffer_deg']

SITE_CONFIG['point'] = ee.Geometry.Point([_lon, _lat])
SITE_CONFIG['region'] = ee.Geometry.Rectangle([
    _lon - _buf, _lat - _buf,
    _lon + _buf, _lat + _buf,
])
SITE_CONFIG['start_date'] = f"{START_YEAR}-01-01"
SITE_CONFIG['end_date']   = f"{END_YEAR}-12-31"

os.makedirs(SITE_CONFIG['output_dir'], exist_ok=True)
print(f"Site:   {SITE_CONFIG['site_name']}  ({SITE_CONFIG['description']})")
print(f"Period: {START_YEAR}–{END_YEAR}")
print(f"Pixel:  lon={_lon}, lat={_lat}")
print(f"Region: [{_lon-_buf:.4f}, {_lat-_buf:.4f}] → [{_lon+_buf:.4f}, {_lat+_buf:.4f}]  (buffer ±{_buf}°)")
print(f"Figures → {os.path.abspath(SITE_CONFIG['output_dir'])}")

---
## Core Helper Functions
These functions standardize HLS loading, Fmask masking, band harmonization, and index generation.

You normally do not need to edit them unless you are adapting the workflow to a different sensor or spectral index set.


In [ ]:
# Import reusable HLS loading helpers from the package module.
from gee_hls.notebook_helpers import (
    COMMON_BANDS,
    L8_BANDS_IN,
    S2_BANDS_IN,
    add_indices,
    load_hls_collections,
    mask_hls,
    prepare_l30,
    prepare_s30,
)

print('Imported HLS loading helpers from gee_hls.notebook_helpers.')

---
## Step 1 — Load Collections and Count Observations
Load `L30`, `S30`, and merged `HLS` collections for the selected site and time range, then summarize unique cloud-free observation days.


In [ ]:
from gee_hls.notebook_helpers import get_dates

print(f'Loading HLS collections for {SITE_CONFIG["site_name"]}...')
l30_col, s30_col, hls_col = load_hls_collections(SITE_CONFIG)

l30_dates = get_dates(l30_col)
s30_dates = get_dates(s30_col)
hls_dates = get_dates(hls_col)

n_l30 = len({d.strftime('%Y-%m-%d') for d in l30_dates})
n_s30 = len({d.strftime('%Y-%m-%d') for d in s30_dates})
n_hls = len({d.strftime('%Y-%m-%d') for d in hls_dates})

print(f'\nCloud-free observation days over study region (unique dates):')
print(f'  L30 (Landsat 8/9):        {n_l30:>5d}')
print(f'  S30 (Sentinel-2 A/B):     {n_s30:>5d}')
print(f'  HLS Combined (L30+S30):   {n_hls:>5d}')
print(f'  HLS / L30-only ratio:     {n_hls/max(n_l30,1):.1f}x')

print(f'\nDate ranges:')
for label, dates in [('L30', l30_dates), ('S30', s30_dates), ('HLS', hls_dates)]:
    if dates:
        print(f'  {label}: {min(dates).date()} → {max(dates).date()}')


---
## Figure 1 — Annual Cloud-Free Observation Frequency
This figure provides the main observation-density comparison and should be the first diagnostic you inspect for a new site.


In [ ]:
from gee_hls.notebook_helpers import count_by_year

years = list(range(SITE_CONFIG['start_year'], SITE_CONFIG['end_year'] + 1))
_, l30_annual = count_by_year(l30_dates, SITE_CONFIG['start_year'], SITE_CONFIG['end_year'])
_, s30_annual = count_by_year(s30_dates, SITE_CONFIG['start_year'], SITE_CONFIG['end_year'])
_, hls_annual = count_by_year(hls_dates, SITE_CONFIG['start_year'], SITE_CONFIG['end_year'])

x = np.arange(len(years))
w = 0.28

fig, ax = plt.subplots(figsize=(12, 5))
bars_l = ax.bar(x - w, l30_annual, width=w, label='L30 (Landsat 8/9)', color='red', alpha=0.85, edgecolor='white')
bars_s = ax.bar(x, s30_annual, width=w, label='S30 (Sentinel-2)', color='#1565C0', alpha=0.85, edgecolor='white')
bars_h = ax.bar(x + w, hls_annual, width=w, label='HLS Combined (L30+S30)', color='#212121', alpha=0.85, edgecolor='white')

for i, (l, h) in enumerate(zip(l30_annual, hls_annual)):
    if l > 0:
        ratio = h / l
        ax.text(x[i] + w, h + 0.5, f'{ratio:.1f}×', ha='center', va='bottom', fontsize=8, color='#424242', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Cloud-Free Observations', fontsize=12)
ax.set_title(
    f'{SITE_CONFIG["site_name"]} — Annual Cloud-Free Observation Frequency\n'
    f'(Number shown above bars = HLS / L30-only ratio)',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=11)
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig1_path = os.path.join(SITE_CONFIG['output_dir'], f"{SITE_CONFIG['site_name']}_fig1_obs_frequency.png")
fig.savefig(fig1_path, dpi=150, bbox_inches='tight')
print(f'Fig 1 saved → {fig1_path}')
plt.show()


---
## CCDC Utility Functions
These helpers convert dates, evaluate harmonic fits, extract point time series, and prepare the pixel-level CCDC plots.


In [ ]:
# Import reusable time-conversion and pixel-level CCDC helpers.
from gee_hls.notebook_helpers import (
    decimal_to_datetime,
    get_pixel_ts,
    harmonic_model,
    run_ccdc_at_point,
    to_decimal_year,
)

print('Imported pixel-level CCDC helpers from gee_hls.notebook_helpers.')

---
## Step 2 — Run Pixel-Level CCDC and Extract Time Series
This section runs CCDC at the study pixel for `L30`, `S30`, and `HLS`, then pulls the raw observation series used in the temporal plots.

Expect several Earth Engine calls here. For most sites this takes a few minutes.


In [ ]:
point       = SITE_CONFIG['point']
plot_bands  = SITE_CONFIG['plot_bands']   # ['NDVI', 'NBR']

# --- Run CCDC at the study pixel for each collection ---
print('Running CCDC at pixel (L30)...')
ccdc_l30 = run_ccdc_at_point(l30_col, point, SITE_CONFIG)

print('Running CCDC at pixel (S30)...')
ccdc_s30 = run_ccdc_at_point(s30_col, point, SITE_CONFIG)

print('Running CCDC at pixel (HLS combined)...')
ccdc_hls = run_ccdc_at_point(hls_col, point, SITE_CONFIG)

# --- Extract raw time series at the study pixel ---
print('Extracting pixel time series...')
ts_l30 = get_pixel_ts(l30_col, point, plot_bands)
ts_s30 = get_pixel_ts(s30_col, point, plot_bands)
ts_hls = get_pixel_ts(hls_col, point, plot_bands)

print(f'\nPixel obs counts — L30: {len(ts_l30)}, S30: {len(ts_s30)}, HLS: {len(ts_hls)}')

# Summarize detected breakpoints
for label, ccdc in [('L30', ccdc_l30), ('S30', ccdc_s30), ('HLS', ccdc_hls)]:
    n_segs = len(ccdc.get('tStart', [])) if ccdc else 0
    breaks = [decimal_to_datetime(t).strftime('%Y-%m')
              for t in ccdc.get('tBreak', []) if t and t > 0] if ccdc else []
    print(f'  {label}: {n_segs} segments, breakpoints at {breaks}')

---
## Figure 2 — Pixel Time Series, CCDC Fits, and Representative Snapshots
This section creates the core temporal interpretation figures:
- the multi-panel time-series plot with sensor-specific fits
- the representative clear-sky snapshot figure tied to detected break periods

Together these show both the temporal sampling advantage of HLS and the visual context around disturbance timing.


In [ ]:
from gee_hls.notebook_helpers import (
    SEG_COLORS,
    add_smooth_fit as _add_smooth_fit,
    get_ccdc_coefs as _get_coefs,
    plot_band_figure,
)

print('CCDC coefficient check (segment 0):')
for label, ccdc in [('L30', ccdc_l30), ('S30', ccdc_s30), ('HLS', ccdc_hls)]:
    for band in plot_bands:
        c = _get_coefs(ccdc, band, 0)
        if c:
            print(f'  {label:4s} {band}: intercept={c[0]:.4f}  slope={c[1]:+.5f}  '
                  f'cos1={c[2]:+.4f}  sin1={c[3]:+.4f}  '
                  f'[harmonic amp ≈ {np.hypot(c[2], c[3]):.4f}]')
        else:
            print(f'  {label:4s} {band}: coefs not found — check collection/CCDC run')

DATASET_STYLES = [
    ('L30 (Landsat 8/9)',        ccdc_l30, ts_l30, '#1565C0'),
    ('S30 (Sentinel-2)',         ccdc_s30, ts_s30, '#2E7D32'),
    ('HLS Combined (L30 + S30)', ccdc_hls, ts_hls, '#212121'),
]

fig2_paths = {}
for band in plot_bands:
    fig = plot_band_figure(DATASET_STYLES, band, SITE_CONFIG)
    path = os.path.join(SITE_CONFIG['output_dir'],
                        f"{SITE_CONFIG['site_name']}_fig2_{band}.png")
    fig.savefig(path, dpi=150, bbox_inches='tight')
    fig2_paths[band] = path
    print(f'Fig 2 ({band}) saved → {path}')
    plt.show()

In [ ]:
# ── Snapshot figure: time series + 3 representative RGB images ───────────────
# Date selection logic:
#   1. Identify CCDC break dates to split the timeline into segments.
#   2. Within each segment, pick the HLS observation that is most temporally
#      isolated — i.e. maximises the distance to its nearest neighbour.
#      These are the dates that are most "useful to fill a gap".
#   3. Three dates are chosen: one pre-break, one near the break, one post-break.
# The selected dates are highlighted with circles on the time series and
# shown as RGB snapshots in the bottom row.

# Ensure anim_roi is defined even if animation cell was not run
_buf = SITE_CONFIG['region_buffer_deg']/10
anim_roi = ee.Geometry.Rectangle([
    SITE_CONFIG['pixel_lon'] - _buf, SITE_CONFIG['pixel_lat'] - _buf,
    SITE_CONFIG['pixel_lon'] + _buf, SITE_CONFIG['pixel_lat'] + _buf,
])

import urllib.request as _urlreq
import io as _io
from PIL import Image as _PIL
_RAW_COL_ID = {
    'L30': 'NASA/HLS/HLSL30/v002',
    'S30': 'NASA/HLS/HLSS30/v002',
}
_COMPOSITE_BANDS = {
    'true_color':  {'L30': ['B4', 'B3', 'B2'], 'S30': ['B4', 'B3', 'B2']},
    'false_color': {'L30': ['B5', 'B4', 'B3'], 'S30': ['B8A', 'B4', 'B3']},
}
_COMPOSITE_VIS = {
    'true_color':  {'min': 0.0, 'max': 0.15, 'gamma': 1.8},
    'false_color': {'min': 0.0, 'max': 0.6,  'gamma': 1.4},
}
THUMB_PX = 256

PLOT_START = None   # override if needed; None = use SITE_CONFIG start_year
PLOT_END   = None   # override if needed; None = use SITE_CONFIG end_year
SNAP_BAND      = plot_bands[0]   # band for the time series panel
from gee_hls.notebook_helpers import (
    MOVING_AVG_MIN_PERIODS,
    MOVING_AVG_WINDOW_DAYS,
    build_moving_average_fit as _build_moving_average_fit,
)
SNAP_COMPOSITE = 'true_color'    # 'true_color' or 'false_color'
SNAP_DPI       = 150
SNAP_OUT = os.path.join(SITE_CONFIG['output_dir'],
                        f"{SITE_CONFIG['site_name']}_snapshot_{SNAP_BAND}.png")


# ── Helper: most-isolated observation within a date window ───────────────────
def _gap_representative(hls_dates_sorted, start_dt, end_dt):
    """Return the date in [start_dt, end_dt] with the largest max-gap to
    its immediate neighbours (prev/next observation in the full sorted list).
    """
    window = [d for d in hls_dates_sorted if start_dt <= d <= end_dt]
    if not window:
        return None
    if len(window) == 1:
        return window[0]
    all_d = hls_dates_sorted
    best, best_gap = None, -1
    for d in window:
        idx = all_d.index(d)
        prev_gap = (d - all_d[idx - 1]).days if idx > 0         else 0
        next_gap = (all_d[idx + 1] - d).days if idx < len(all_d)-1 else 0
        gap = max(prev_gap, next_gap)
        if gap > best_gap:
            best_gap, best = gap, d
    return best


# ── Step 1: Build sorted list of all unique HLS observation dates ─────────────
_hls_ts = ts_hls[["date", SNAP_BAND]].dropna().copy()
_hls_ts["date"] = pd.to_datetime(_hls_ts["date"]).dt.normalize()
_all_hls_dates = sorted(_hls_ts["date"].unique().tolist())

_l30_ts = ts_l30[["date", SNAP_BAND]].dropna().copy()
_l30_ts["date"] = pd.to_datetime(_l30_ts["date"]).dt.normalize()
_s30_ts = ts_s30[["date", SNAP_BAND]].dropna().copy()
_s30_ts["date"] = pd.to_datetime(_s30_ts["date"]).dt.normalize()


_n_l30_snap = len(_l30_ts)
_n_s30_snap = len(_s30_ts)
_n_hls_snap = len(_hls_ts)


# ── Step 2: Select three snapshot dates ──────────────────────────────────────
# Strategy:
#   - If breaks are detected, prioritize one representative sample per break.
#   - With two breaks, use: pre-disturbance baseline + break 1 + break 2.
#   - With one break, use: pre-disturbance baseline + break 1 + post-disturbance.
#   - If there are more than three breaks, sample up to three break windows.
#   - If there are no breaks, fall back to early / middle / late timeline samples.

_l30_date_set = {d.strftime("%Y-%m-%d") for d in l30_dates}
_s30_date_set = {d.strftime("%Y-%m-%d") for d in s30_dates}

# Dates that belong exclusively to each sensor in the HLS time series
_l30_only = [d for d in _all_hls_dates if d.strftime("%Y-%m-%d") in _l30_date_set
             and d.strftime("%Y-%m-%d") not in _s30_date_set]
_s30_only  = [d for d in _all_hls_dates if d.strftime("%Y-%m-%d") in _s30_date_set
              and d.strftime("%Y-%m-%d") not in _l30_date_set]

# Clamp selection window to PLOT_START / PLOT_END when defined
_win_s = pd.Timestamp(PLOT_START) if PLOT_START else pd.Timestamp(f"{SITE_CONFIG['start_year']}-01-01")
_win_e = pd.Timestamp(PLOT_END)   if PLOT_END   else pd.Timestamp(f"{SITE_CONFIG['end_year']}-12-31")

# Re-filter date lists to the visible window
_all_hls_dates = [d for d in _all_hls_dates if _win_s <= d <= _win_e]
_l30_only      = [d for d in _l30_only      if _win_s <= d <= _win_e]
_s30_only      = [d for d in _s30_only      if _win_s <= d <= _win_e]

_break_dts = sorted([
    pd.Timestamp(decimal_to_datetime(t)).normalize()
    for t in (ccdc_hls.get("tBreak") or []) if t and t > 0
    if _win_s <= pd.Timestamp(decimal_to_datetime(t)).normalize() <= _win_e
])
_t_start = _all_hls_dates[0]  if _all_hls_dates else _win_s
_t_end   = _all_hls_dates[-1] if _all_hls_dates else _win_e

SNAP_MAX_PANELS = 3
SNAP_BREAK_CONTEXT_DAYS = 120


def _window_midpoint(win_s, win_e):
    return win_s + (win_e - win_s) / 2


def _best_date_near_target(target_dt, win_s, win_e, prefer_sensor="S30"):
    """Return (date, sensor) closest to target_dt within the given window.

    Prefers sensor-exclusive dates first, then the other sensor, then any HLS date.
    """
    primary   = _s30_only if prefer_sensor == "S30" else _l30_only
    secondary = _l30_only if prefer_sensor == "S30" else _s30_only
    pools = [
        (primary, prefer_sensor),
        (secondary, "L30" if prefer_sensor == "S30" else "S30"),
        (_all_hls_dates, None),
    ]
    for pool, sen in pools:
        in_window = [d for d in pool if win_s <= d <= win_e]
        if not in_window:
            continue
        in_window = sorted(
            in_window,
            key=lambda d: (abs((d - target_dt).days), d)
        )
        d = in_window[0]
        if sen is None:
            sen = "L30" if d.strftime("%Y-%m-%d") in _l30_date_set else "S30"
        return d, sen
    return None, None


# Build segments: boundaries are [t_start, break_1, break_2, ..., t_end]
_boundaries = [_t_start] + _break_dts + [_t_end]
_segments   = list(zip(_boundaries[:-1], _boundaries[1:]))   # (seg_start, seg_end)
n_segs = len(_segments)
n_breaks = len(_break_dts)

snapshot_specs = []

if n_breaks == 0:
    # No breakpoints: sample early, middle, and late periods.
    thirds = np.linspace(0, 2, SNAP_MAX_PANELS).astype(int)
    for idx, frac in enumerate([0.17, 0.50, 0.83]):
        target_dt = _t_start + (_t_end - _t_start) * frac
        d, sen = _best_date_near_target(target_dt, _t_start, _t_end,
                                        prefer_sensor="L30" if idx == 0 else "S30")
        snapshot_specs.append({
            "label": f"Period {idx + 1}",
            "date": d,
            "sensor": sen,
            "win_start": _t_start,
            "win_end": _t_end,
        })
elif n_breaks == 1:
    pre_s, pre_e = _segments[0]
    post_s, post_e = _segments[-1]
    pre_target = _window_midpoint(pre_s, pre_e)
    break_dt = _break_dts[0]
    break_s = max(pre_s, break_dt - pd.Timedelta(days=SNAP_BREAK_CONTEXT_DAYS))
    break_e = min(post_e, break_dt + pd.Timedelta(days=SNAP_BREAK_CONTEXT_DAYS))
    post_target = _window_midpoint(post_s, post_e)

    for label, target_dt, win_s, win_e, prefer in [
        ("Pre-disturbance", pre_target, pre_s, pre_e, "L30"),
        ("Break 1", break_dt, break_s, break_e, "S30"),
        ("Post-disturbance", post_target, post_s, post_e, "S30"),
    ]:
        d, sen = _best_date_near_target(target_dt, win_s, win_e, prefer_sensor=prefer)
        snapshot_specs.append({
            "label": label,
            "date": d,
            "sensor": sen,
            "win_start": win_s,
            "win_end": win_e,
        })
elif n_breaks == 2:
    pre_s, pre_e = _segments[0]
    pre_target = _window_midpoint(pre_s, pre_e)
    d, sen = _best_date_near_target(pre_target, pre_s, pre_e, prefer_sensor="L30")
    snapshot_specs.append({
        "label": "Pre-disturbance",
        "date": d,
        "sensor": sen,
        "win_start": pre_s,
        "win_end": pre_e,
    })

    for break_idx, break_dt in enumerate(_break_dts, start=1):
        left_bound = _t_start if break_idx == 1 else _break_dts[break_idx - 2]
        right_bound = _t_end if break_idx == n_breaks else _break_dts[break_idx]
        win_s = max(left_bound, break_dt - pd.Timedelta(days=SNAP_BREAK_CONTEXT_DAYS))
        win_e = min(right_bound, break_dt + pd.Timedelta(days=SNAP_BREAK_CONTEXT_DAYS))
        d, sen = _best_date_near_target(break_dt, win_s, win_e, prefer_sensor="S30")
        snapshot_specs.append({
            "label": f"Break {break_idx}",
            "date": d,
            "sensor": sen,
            "win_start": win_s,
            "win_end": win_e,
        })
else:
    chosen_break_ids = sorted({
        int(round(v)) for v in np.linspace(0, n_breaks - 1, SNAP_MAX_PANELS)
    })
    for out_idx, break_id in enumerate(chosen_break_ids[:SNAP_MAX_PANELS], start=1):
        break_dt = _break_dts[break_id]
        left_bound = _t_start if break_id == 0 else _break_dts[break_id - 1]
        right_bound = _t_end if break_id == n_breaks - 1 else _break_dts[break_id + 1]
        win_s = max(left_bound, break_dt - pd.Timedelta(days=SNAP_BREAK_CONTEXT_DAYS))
        win_e = min(right_bound, break_dt + pd.Timedelta(days=SNAP_BREAK_CONTEXT_DAYS))
        d, sen = _best_date_near_target(break_dt, win_s, win_e, prefer_sensor="S30")
        snapshot_specs.append({
            "label": f"Break {break_id + 1}",
            "date": d,
            "sensor": sen,
            "win_start": win_s,
            "win_end": win_e,
        })

# Keep three panels; if fewer snapshots were found, pad with empty slots.
snapshot_specs = snapshot_specs[:SNAP_MAX_PANELS]
while len(snapshot_specs) < SNAP_MAX_PANELS:
    snapshot_specs.append({
        "label": f"Period {len(snapshot_specs) + 1}",
        "date": None,
        "sensor": "S30",
        "win_start": _t_start,
        "win_end": _t_end,
    })

snap_dates = [spec["date"].strftime("%Y-%m-%d") if spec["date"] is not None else None
              for spec in snapshot_specs]
snap_sensors = [spec["sensor"] for spec in snapshot_specs]
snap_windows = [(spec["win_start"], spec["win_end"]) for spec in snapshot_specs]
snap_labels = [f"{spec['label']} ({spec['sensor']})" for spec in snapshot_specs]

print(f"Snapshot dates  : {snap_dates}")
print(f"Snapshot sensors: {snap_sensors}")


# ── Step 3: Fetch RGB thumbnails for snapshot dates ───────────────────────────
SNAP_CLEAR_THRESHOLD = 0.95   # require at least 95% valid clear-sky pixels

def _fetch_snap_thumb(date_str, sensor, roi, win_start=None, win_end=None):
    """Fetch a representative clear-sky thumbnail near date_str.

    Selection logic within the allowed break-focused window:
      1. Only keep observations with at least SNAP_CLEAR_THRESHOLD valid pixels.
      2. Among those clear observations, choose the one closest to the target date.
      3. If no sufficiently clear observation exists in the window, return None.

    Returns (PIL image or None, actual_date or None).
    """
    if date_str is None:
        return None, None

    col_id = _RAW_COL_ID[sensor]
    bands  = _COMPOSITE_BANDS[SNAP_COMPOSITE][sensor]
    vis    = _COMPOSITE_VIS[SNAP_COMPOSITE]

    sensor_dates = sorted({d.strftime("%Y-%m-%d") for d in
                           (l30_dates if sensor == "L30" else s30_dates)})
    target = pd.Timestamp(date_str)
    win_start = pd.Timestamp(win_start) if win_start is not None else target
    win_end = pd.Timestamp(win_end) if win_end is not None else target

    candidates = []
    for d in sensor_dates:
        ts = pd.Timestamp(d)
        if ts < win_start or ts > win_end:
            continue
        candidates.append(d)

    if not candidates:
        return None, None

    def _get_valid_frac(best):
        """Return (img_masked, valid_frac) for a given date string, or (None, 0)."""
        next_day = (pd.Timestamp(best) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        try:
            img_full = (ee.ImageCollection(col_id)
                        .filterDate(best, next_day)
                        .filterBounds(roi)
                        .mosaic())
            fmask  = img_full.select("Fmask")
            clear  = fmask.eq(64).Or(fmask.eq(128))
            masked = img_full.updateMask(clear).select(bands)
            frac   = (
                masked.select(bands[0]).mask()
                .reduceRegion(ee.Reducer.mean(), roi, scale=60, bestEffort=True)
                .getInfo().get(bands[0], 0) or 0
            )
            return masked, frac
        except Exception as e:
            if "no bands" not in str(e).lower():
                print(f"  Warning: {sensor} {best}: {e}")
            return None, 0

    def _render(img_masked, best):
        """Download thumbnail; return PIL image or None if blank."""
        try:
            img = img_masked.unmask(0)
            url = img.getThumbURL({
                "bands": bands, **vis,
                "region": roi, "dimensions": THUMB_PX, "format": "png",
            })
            with _urlreq.urlopen(url, timeout=30) as resp:
                pil_img = _PIL.open(_io.BytesIO(resp.read())).convert("RGB")
            arr = np.array(pil_img, dtype=np.float32)
            if arr.mean() > 245 or arr.std() < 3:
                return None
            return pil_img
        except Exception as e:
            print(f"  Warning: {sensor} render {best}: {e}")
            return None

    clear_candidates = []
    for best in candidates:
        img_masked, frac = _get_valid_frac(best)
        if img_masked is None:
            continue
        if float(frac) < SNAP_CLEAR_THRESHOLD:
            continue
        clear_candidates.append({
            "date": best,
            "frac": float(frac),
            "dist": int(abs((pd.Timestamp(best) - target).days)),
            "img": img_masked,
        })

    if not clear_candidates:
        print(f"  [{sensor} {date_str}] no >= {SNAP_CLEAR_THRESHOLD*100:.0f}% clear image found "
              f"between {win_start.strftime('%Y-%m-%d')} and {win_end.strftime('%Y-%m-%d')}")
        return None, None

    clear_candidates.sort(key=lambda item: (item["dist"], -item["frac"], item["date"]))
    for item in clear_candidates:
        pil = _render(item["img"], item["date"])
        if pil is not None:
            return pil, item["date"]

    return None, None


print("Fetching RGB snapshots...")
snap_thumbs = {}   # {date_str: {'L30': (pil, actual_date), 'S30': (pil, actual_date)}}
for sd, (win_s, win_e) in zip(snap_dates, snap_windows):
    snap_thumbs[sd] = {
        "L30": _fetch_snap_thumb(sd, "L30", anim_roi, win_start=win_s, win_end=win_e),
        "S30": _fetch_snap_thumb(sd, "S30", anim_roi, win_start=win_s, win_end=win_e),
    }
    print(f"  {sd}: L30={snap_thumbs[sd]['L30'][1]}  S30={snap_thumbs[sd]['S30'][1]}")


# ── Step 4: Build figure ──────────────────────────────────────────────────────
_snap_colors = ["#1B5E20", "#B71C1C", "#0D47A1"]   # green / red / blue per period

fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1.6, 1],
                        hspace=0.38, wspace=0.12)
ax_ts = fig.add_subplot(gs[0, :])

# ── Time series ───────────────────────────────────────────────────────────────
def _ds(d):
    return pd.Timestamp(d).strftime("%Y-%m-%d")

ax_ts.scatter(_l30_ts["date"], _l30_ts[SNAP_BAND],
              c="red", s=20, alpha=0.55, marker="o",
              label=f"L30 (Landsat, n={_n_l30_snap})", zorder=2)
ax_ts.scatter(_s30_ts["date"], _s30_ts[SNAP_BAND],
              c="#1565C0", s=20, alpha=0.55, marker="^",
              label=f"S30 (Sentinel-2, n={_n_s30_snap})", zorder=2)

# Moving-average fit over full period
_fit_d, _fit_v = _build_moving_average_fit(
    ts_hls,
    SNAP_BAND,
    _xlim_s,
    _xlim_e,
)
if len(_fit_d):
    ax_ts.plot(_fit_d, _fit_v, "--", color="black", lw=1.8, alpha=0.5,
               label=(f"HLS moving average ({MOVING_AVG_WINDOW_DAYS}-day, "
                      f"n={_n_hls_snap})"), zorder=3)

# Break date lines
for bp in (ccdc_hls.get("tBreak") or []):
    if bp and bp > 0:
        ax_ts.axvline(decimal_to_datetime(bp), color="orange",
                      lw=1.4, linestyle="--", alpha=0.7, zorder=4)

# Circle highlights for snapshot dates
for i, sd in enumerate(snap_dates):
    hit_l = _l30_ts[_l30_ts["date"].apply(_ds) == sd]
    hit_s = _s30_ts[_s30_ts["date"].apply(_ds) == sd]
    hit   = hit_l if not hit_l.empty else hit_s
    color = _snap_colors[i]
    if not hit.empty:
        ax_ts.scatter(hit["date"], hit[SNAP_BAND], s=220,
                      marker="o", facecolors="none",
                      edgecolors=color, linewidths=2.2, zorder=7)


_xlim_s = pd.Timestamp(PLOT_START) if PLOT_START else pd.Timestamp(f"{SITE_CONFIG['start_year']}-01-01")
_xlim_e = pd.Timestamp(PLOT_END)   if PLOT_END   else pd.Timestamp(f"{SITE_CONFIG['end_year']}-12-31")
ax_ts.set_xlim(_xlim_s, _xlim_e)
_all_v = pd.concat([_l30_ts[SNAP_BAND], _s30_ts[SNAP_BAND]]).dropna()
_yp = (_all_v.max() - _all_v.min()) * 0.12
ax_ts.set_ylim(_all_v.min() - _yp, _all_v.max() + _yp)
ax_ts.xaxis.set_major_locator(mdates.YearLocator())
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_ts.set_ylabel(SNAP_BAND, fontsize=11)
ax_ts.set_title(
    f"{SITE_CONFIG['site_name']}  —  {SNAP_BAND} time series with representative snapshots\n"
    f"lon={SITE_CONFIG['pixel_lon']:.2f}, lat={SITE_CONFIG['pixel_lat']:.2f}  |  HLS total observations: {_n_hls_snap}",
    fontsize=12, fontweight="bold"
)
ax_ts.legend(loc="lower left", fontsize=13.5, framealpha=0.8, markerscale=1.4, borderpad=0.9, labelspacing=0.7, handlelength=2.4)
ax_ts.grid(False)
ax_ts.spines["top"].set_visible(False)
ax_ts.spines["right"].set_visible(False)


# ── RGB snapshot panels ───────────────────────────────────────────────────────
from matplotlib.patches import Rectangle as _Rect, Circle as _Circ

for col_idx, (sd, slabel, scolor) in enumerate(zip(snap_dates, snap_labels, _snap_colors)):
    ax_img = fig.add_subplot(gs[1, col_idx])
    ax_img.axis("off")

    # Use the sensor that was explicitly selected for this snapshot;
    # fall back to the other sensor only if the primary has no image.
    _primary = snap_sensors[col_idx]
    _fallback = "S30" if _primary == "L30" else "L30"
    pil_img, actual_date, sensor_used = None, None, None
    for sensor in [_primary, _fallback]:
        _p, _d = snap_thumbs[sd][sensor]
        if _p is not None:
            pil_img, actual_date, sensor_used = _p, _d, sensor
            break

    if pil_img is not None:
        ax_img.imshow(pil_img)
        # Yellow square at study pixel centre
        cx, cy = THUMB_PX / 2, THUMB_PX / 2
        sz = max(6, THUMB_PX // 32)
        ax_img.add_patch(_Rect((cx - sz/2, cy - sz/2), sz, sz,
                               linewidth=2, edgecolor="yellow",
                               facecolor="none", zorder=5))
        ax_img.set_title(
            f"{slabel}\n{sensor_used}: {actual_date}",
            fontsize=9, color=scolor, fontweight="bold"
        )
    else:
        ax_img.text(0.5, 0.5, f"No image\navailable near\n{sd}",
                    ha="center", va="center", transform=ax_img.transAxes,
                    fontsize=10, color="gray", style="italic")
        ax_img.set_title(slabel, fontsize=9, color=scolor, fontweight="bold")

    # Coloured border matching the circle on the time series
    for spine in ax_img.spines.values():
        spine.set_edgecolor(scolor)
        spine.set_linewidth(2)

plt.tight_layout()
fig.savefig(SNAP_OUT, dpi=SNAP_DPI, bbox_inches="tight")
print(f"Snapshot figure saved → {SNAP_OUT}")
plt.show()

---
## Optional Figure 2c — Time-Series + RGB Animation
Create an animated visualization of the HLS time series and RGB imagery through time.

This section is optional and is most useful for presentations or qualitative review.


In [ ]:
import urllib.request as _urlreq
import io as _io
from PIL import Image as _PIL

# ── Settings (edit as needed) ─────────────────────────────────────────────────
ANIM_BAND     = plot_bands[0]   # band shown in time-series panel (e.g. 'NDVI')
ANIM_ROI_BUF  = SITE_CONFIG['region_buffer_deg'] # inherit spatial buffer from SITE_CONFIG
THUMB_PX      = 256             # thumbnail pixel dimensions (GEE renders this)
FRAMES_DPI    = 120             # DPI for exported PNGs
FORCE_EXPORT  = False           # True to re-export already-saved frames
PLOT_START    = None           # e.g. '2018-01-01'; None = use SITE_CONFIG start_year
PLOT_END      = None             # e.g. '2023-12-31'; None = use SITE_CONFIG end_year

# ── Derived objects ───────────────────────────────────────────────────────────
_lon, _lat = SITE_CONFIG['pixel_lon'], SITE_CONFIG['pixel_lat']
anim_roi = ee.Geometry.Rectangle([
    _lon - ANIM_ROI_BUF, _lat - ANIM_ROI_BUF,
    _lon + ANIM_ROI_BUF, _lat + ANIM_ROI_BUF,
])
frames_dir = os.path.join(SITE_CONFIG['output_dir'],
                           f"{SITE_CONFIG['site_name']}_frames_{ANIM_BAND}")
os.makedirs(frames_dir, exist_ok=True)

# Raw (unmasked) HLS collection IDs — used for true-colour thumbnails.
# We intentionally skip cloud-masking here: the prepared l30_col/s30_col have
# updateMask() applied, making most pixels transparent → black in PNG output.
# Using the raw collections + unmask(0) gives a proper true-colour composite.
_RAW_COL_ID = {
    'L30': 'NASA/HLS/HLSL30/v002',
    'S30': 'NASA/HLS/HLSS30/v002',
}

# ── On-demand RGB thumbnail (no bulk download) ────────────────────────────────
# COMPOSITE_TYPE controls which bands are shown in the RGB panels.
#   'true_color'  : natural colour  (Red / Green / Blue)
#   'false_color' : NIR / Red / Green  — vegetation bright red, great for forest
COMPOSITE_TYPE = 'true_color'

_COMPOSITE_BANDS = {
    'true_color':  {'L30': ['B4', 'B3', 'B2'], 'S30': ['B4', 'B3', 'B2']},
    'false_color': {'L30': ['B5', 'B4', 'B3'], 'S30': ['B8A', 'B4', 'B3']},
}
_COMPOSITE_VIS = {
    # GEE handles brightening server-side — no post-processing stretch needed.
    # true_color : forest ~0.02-0.10 in visible → fits well in max=0.15, gamma=1.8
    # false_color: NIR    ~0.30-0.50          → needs wider max=0.6
    'true_color':  {'min': 0.0, 'max': 0.15, 'gamma': 1.8},
    'false_color': {'min': 0.0, 'max': 0.6,  'gamma': 1.4},
}

_thumb_cache = {}   # {(label, date_str): PIL.Image or None}

def _fetch_thumb(col_dates, date_str, roi, label):
    """Return a PIL Image only if this sensor has an observation on date_str.

    Each frame shows the image acquired on that exact date for each sensor.
    If the sensor has no observation on date_str, returns (None, None) so the
    panel shows 'No data' instead of a stale image from a prior date.
    """
    # Only proceed if this sensor actually acquired on this calendar date
    date_set = {d.strftime('%Y-%m-%d') for d in col_dates}
    if date_str not in date_set:
        return None, None
    best = date_str

    cache_key = (COMPOSITE_TYPE, label, best)
    if cache_key in _thumb_cache:
        return _thumb_cache[cache_key], best

    try:
        next_day = (pd.Timestamp(best) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        bands = _COMPOSITE_BANDS[COMPOSITE_TYPE][label]
        vis   = _COMPOSITE_VIS[COMPOSITE_TYPE]
        img = (ee.ImageCollection(_RAW_COL_ID[label])
               .filterDate(best, next_day)
               .filterBounds(roi)
               .mosaic()
               .select(bands)
               .unmask(0))
        url = img.getThumbURL({
            'bands': bands,
            **vis,
            'region': roi,
            'dimensions': THUMB_PX,
            'format': 'png',
        })
        with _urlreq.urlopen(url, timeout=30) as resp:
            pil_img = _PIL.open(_io.BytesIO(resp.read())).convert('RGB')
        # Reject blank images: GEE returns an all-white or near-uniform PNG
        # for empty/all-cloud scenes rather than raising an error.
        _arr = np.array(pil_img, dtype=np.float32)
        if _arr.mean() > 245 or _arr.std() < 3:
            _thumb_cache[cache_key] = None
            return None, best
        _thumb_cache[cache_key] = pil_img
        return pil_img, best
    except Exception as e:
        print(f'  Warning: {label} thumb {best}: {e}')
        _thumb_cache[cache_key] = None
        return None, best


# ── Step 1: Prepare time-series data and HLS smooth fit ───────────────────────
# Normalize observation datetimes to midnight so that <= comparisons with
# pd.Timestamp(date_str) (which is also midnight) work correctly.
# Without this, an obs at 2019-04-28 14:32 UTC fails the <= '2019-04-28' check.
ts_anim_l30 = ts_l30[['date', ANIM_BAND]].dropna().copy()
ts_anim_s30 = ts_s30[['date', ANIM_BAND]].dropna().copy()
ts_anim_l30['date'] = pd.to_datetime(ts_anim_l30['date']).dt.normalize()
ts_anim_s30['date'] = pd.to_datetime(ts_anim_s30['date']).dt.normalize()

def _build_smooth_fit(ts_df, ccdc, band, start_yr, end_yr, smooth_factor=0.4):
    from scipy.interpolate import UnivariateSpline
    df = ts_df[['date', band]].dropna().copy()
    df['t'] = df['date'].apply(to_decimal_year)
    df = df.sort_values('t').reset_index(drop=True)
    t_all = np.linspace(start_yr, end_yr, 2000)
    vals  = np.full(len(t_all), np.nan)
    for ts_i, te_i in zip(
            (ccdc or {}).get('tStart', []) or [],
            (ccdc or {}).get('tEnd',   []) or []):
        seg = df[(df['t'] >= ts_i) & (df['t'] <= te_i)]
        if len(seg) < 3:
            continue
        x, y = seg['t'].values, seg[band].values
        try:
            s  = len(x) * np.var(y) * smooth_factor
            sp = UnivariateSpline(x, y, k=min(3, len(x) - 1), s=max(s, 1e-9))
            mask = (t_all >= x[0]) & (t_all <= x[-1])
            vals[mask] = sp(t_all[mask])
        except Exception:
            pass
    return [decimal_to_datetime(t) for t in t_all], vals

hls_fit_dates, hls_fit_vals = _build_smooth_fit(
    ts_hls, ccdc_hls, ANIM_BAND,
    SITE_CONFIG['start_year'], SITE_CONFIG['end_year']
)

_all_obs = pd.concat([ts_anim_l30[ANIM_BAND], ts_anim_s30[ANIM_BAND]]).dropna()
_ypad    = (_all_obs.max() - _all_obs.min()) * 0.12
YLIM     = (_all_obs.min() - _ypad, _all_obs.max() + _ypad)
bp_dates = [decimal_to_datetime(t)
            for t in (ccdc_hls.get('tBreak') or []) if t and t > 0]


# ── Step 2: Collect observation dates (filtered to PLOT_START / PLOT_END) ─────
_ts_start = pd.Timestamp(PLOT_START) if PLOT_START else pd.Timestamp(f"{SITE_CONFIG['start_year']}-01-01")
_ts_end   = pd.Timestamp(PLOT_END)   if PLOT_END   else pd.Timestamp(f"{SITE_CONFIG['end_year']}-12-31")

frame_dates = sorted(set(
    [d.strftime('%Y-%m-%d') for d in ts_anim_l30['date'] if _ts_start <= d <= _ts_end] +
    [d.strftime('%Y-%m-%d') for d in ts_anim_s30['date'] if _ts_start <= d <= _ts_end]
))
print(f'{len(frame_dates)} observation dates in [{_ts_start.date()} → {_ts_end.date()}]  →  {frames_dir}/')


# ── Step 3: Export one PNG per observation date ───────────────────────────────
print('Exporting figures (RGB thumbnails fetched on demand from GEE)...')
for k, date_str in enumerate(frame_dates):
    out_path = os.path.join(frames_dir, f'{date_str}.png')
    if not FORCE_EXPORT and os.path.exists(out_path):
        continue

    # Use < next_day so all observations on date_str are included regardless
    # of their UTC time-of-day component (e.g. a Landsat overpass at 14:32 UTC
    # must appear on the frame for that calendar date).
    current_dt = pd.Timestamp(date_str)
    _next_dt   = current_dt + pd.Timedelta(days=1)

    # Pre-fetch thumbnails (cached — no extra GEE cost)
    _pil_l30, _best_l30 = _fetch_thumb(l30_dates, date_str, anim_roi, 'L30')
    _pil_s30, _best_s30 = _fetch_thumb(s30_dates, date_str, anim_roi, 'S30')

    # Fixed 2-panel layout — axes are always at the same positions so the
    # figure never shifts when one sensor has no data.
    fig = plt.figure(figsize=(14, 8))
    gs  = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1.5, 1],
                            hspace=0.35, wspace=0.1)
    ax_ts = fig.add_subplot(gs[0, :])
    ax_l  = fig.add_subplot(gs[1, 0])
    ax_s  = fig.add_subplot(gs[1, 1])

    # --- Time series panel ---
    # Convert to date-string for comparison — robust against any datetime dtype
    # (Python datetime, datetime64, pd.Timestamp) and UTC time-of-day offsets.
    def _date_str(d):
        return pd.Timestamp(d).strftime('%Y-%m-%d')

    l30_vis = ts_anim_l30[ts_anim_l30['date'].apply(_date_str) <= date_str]
    s30_vis = ts_anim_s30[ts_anim_s30['date'].apply(_date_str) <= date_str]

    ax_ts.scatter(l30_vis['date'], l30_vis[ANIM_BAND],
                  c='red', s=22, alpha=0.65, marker='o',
                  label='L30 (Landsat)', zorder=2)
    ax_ts.scatter(s30_vis['date'], s30_vis[ANIM_BAND],
                  c='#1565C0', s=22, alpha=0.65, marker='^',
                  label='S30 (Sentinel-2)', zorder=2)

    # Highlight the observation that matches the image currently shown for each
    # sensor (image title date = _best_l30 / _best_s30).
    for ts_df, best_date, marker, fc, ec in [
        (l30_vis, _best_l30, 'o', 'red',     'darkred'),
        (s30_vis, _best_s30, '^', '#1565C0', '#0d3a6e'),
    ]:
        if best_date is None:
            continue
        hit = ts_df[ts_df['date'].apply(_date_str) == best_date]
        if not hit.empty:
            ax_ts.scatter(hit['date'], hit[ANIM_BAND],
                          s=120, marker=marker, facecolors=fc,
                          edgecolors=ec, linewidths=1.5, zorder=6)

    # Smooth fit clipped to observations available so far
    fit_mask = [d <= current_dt for d in hls_fit_dates]
    fit_dates_vis = [d for d, m in zip(hls_fit_dates, fit_mask) if m]
    fit_vals_vis  = [v for v, m in zip(hls_fit_vals,  fit_mask) if m]
    if fit_dates_vis:
        ax_ts.plot(fit_dates_vis, fit_vals_vis, '--',
                   color='black', lw=1.8, alpha=0.5, label='HLS smooth fit', zorder=3)

    for bp in bp_dates:
        if bp <= current_dt:
            ax_ts.axvline(bp, color='orange', lw=1.5, linestyle='--', alpha=0.7)

    _ts_start = pd.Timestamp(PLOT_START) if PLOT_START else pd.Timestamp(f"{SITE_CONFIG['start_year']}-01-01")
    _ts_end   = pd.Timestamp(PLOT_END)   if PLOT_END   else pd.Timestamp(f"{SITE_CONFIG['end_year']}-12-31")
    ax_ts.set_xlim(_ts_start, _ts_end)
    ax_ts.set_ylim(YLIM)
    ax_ts.xaxis.set_major_locator(mdates.YearLocator())
    ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax_ts.set_ylabel(ANIM_BAND, fontsize=11)
    ax_ts.set_title(f"{SITE_CONFIG['site_name']}  —  {date_str}",
                    fontsize=12, fontweight='bold')
    ax_ts.legend(loc='lower right', fontsize=8, framealpha=0.7)
    ax_ts.grid(False)
    ax_ts.spines['top'].set_visible(False)
    ax_ts.spines['right'].set_visible(False)

    # --- RGB panels: fixed positions, text shown for missing data ---
    for ax, pil_img, best_date, label, color in [
        (ax_l, _pil_l30, _best_l30, 'L30', '#1565C0'),
        (ax_s, _pil_s30, _best_s30, 'S30', '#2E7D32'),
    ]:
        ax.axis('off')
        if pil_img is not None:
            ax.imshow(pil_img)
            star = '\u2605 ' if best_date == date_str else ''
            ax.set_title(f'{star}{label}: {best_date}', fontsize=10,
                         color=color, fontweight='bold')
            # Yellow hollow square marking the study pixel at image centre
            cx, cy = THUMB_PX / 2, THUMB_PX / 2
            sz = max(6, THUMB_PX // 32)   # ~8 px for 256-px thumbnail
            from matplotlib.patches import Rectangle as _Rect
            ax.add_patch(_Rect(
                (cx - sz / 2, cy - sz / 2), sz, sz,
                linewidth=1.8, edgecolor='yellow', facecolor='none', zorder=5
            ))
        else:
            ax.text(0.5, 0.5, f'No {label} data available',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=11, color='gray', style='italic')
            ax.set_title(f'{label}: —', fontsize=10, color='gray')

    fig.savefig(out_path, dpi=FRAMES_DPI, bbox_inches='tight')
    plt.close(fig)

    if (k + 1) % 50 == 0:
        print(f'  {k+1}/{len(frame_dates)} figures saved')

print(f'\nDone. {len(frame_dates)} figures in {frames_dir}/')
print(f'Tip: assemble into video with:')
print(f'  ffmpeg -framerate 5 -pattern_type glob -i "{frames_dir}/*.png" output.mp4')


---
## Step 3 — Spatial CCDC (Asset-Backed or On-the-Fly)
This section builds the spatial disturbance products used in the map and validation steps.

Recommended workflow:
- export `L30`, `S30`, and `HLS` CCDC outputs to Earth Engine assets ahead of time
- set the three asset paths in **Site Configuration**

If assets are not available, the notebook can run on-the-fly for small study areas, but that path is slower and more fragile.


In [ ]:
from gee_hls.notebook_helpers import (
    ccdc_array_to_scalar,
    load_or_compute_ccdc,
    scalar_ccdc_to_gdf,
)

# ── Cache control ─────────────────────────────────────────────────────────────
# Set FORCE_RECOMPUTE = True to re-run GEE and overwrite saved files.
# Set FORCE_RECOMPUTE = False (default) to load from disk if available.
FORCE_RECOMPUTE = False

map_band  = SITE_CONFIG['plot_bands'][1]   # NBR for forest disturbance
map_scale = 250                           # metres; increase if GEE times out
map_tiles = 2                              # NxN tiling to avoid GEE memory limit
                                           # (1=single request, 2=4 tiles, 3=9 tiles)

# ── Cache file paths ──────────────────────────────────────────────────────────
# By default cache files are written to / read from SITE_CONFIG['output_dir'].
# Set CACHE_DIR to any existing directory to load pre-computed results from
# a different location (e.g. a shared drive or a previous run's output folder).
# Set to None to use the default output directory.
CACHE_DIR  = None   # e.g. str(FIGURES_DIR / 'spatial_ccdc' / 'shared_cache') or None

_buf_str   = str(SITE_CONFIG['region_buffer_deg']).replace('.', 'p')  # e.g. 0.055 → '0p055'
_cache_tag = f"{SITE_CONFIG['site_name']}_{map_band}_{map_scale}m_buf{_buf_str}deg"
_cache_dir = CACHE_DIR if CACHE_DIR else SITE_CONFIG['output_dir']
os.makedirs(_cache_dir, exist_ok=True)
cache_hls  = os.path.join(_cache_dir, f'cache_hls_{_cache_tag}.geojson')
cache_s30  = os.path.join(_cache_dir, f'cache_s30_{_cache_tag}.geojson')
cache_l30  = os.path.join(_cache_dir, f'cache_l30_{_cache_tag}.geojson')
print(f'Cache dir : {os.path.abspath(_cache_dir)}')
print(f'Cache tag : {_cache_tag}')

# ── Load from cache or recompute ──────────────────────────────────────────────
# When CACHE_DIR is explicitly set, load the first matching cache_*.geojson
# files found in that directory — ignoring the cache tag (site/scale/buffer).
# This lets you reuse results from a previous run with different settings.
if CACHE_DIR and not FORCE_RECOMPUTE:
    import glob as _glob
    def _find_cache(prefix):
        hits = sorted(_glob.glob(os.path.join(CACHE_DIR, f'cache_{prefix}_*.geojson')))
        return hits[0] if hits else None
    _dir_hls = _find_cache('hls')
    _dir_s30 = _find_cache('s30')
    _dir_l30 = _find_cache('l30')
    if _dir_hls and _dir_s30 and _dir_l30:
        print(f'CACHE_DIR set — loading files directly (tag ignored):')
        for label, path in [('HLS', _dir_hls), ('S30', _dir_s30), ('L30', _dir_l30)]:
            print(f'  {label}: {path}')
        df_hls = gpd.read_file(_dir_hls)
        df_s30 = gpd.read_file(_dir_s30)
        df_l30 = gpd.read_file(_dir_l30)
        print(f'\n  HLS pixels loaded: {len(df_hls)}')
        print(f'  S30 pixels loaded: {len(df_s30)}')
        print(f'  L30 pixels loaded: {len(df_l30)}')
        _all_cached = True
    else:
        print(f'CACHE_DIR set but no cache_*.geojson files found in {CACHE_DIR}')
        _all_cached = False
else:
    _all_cached = (not FORCE_RECOMPUTE
                   and os.path.exists(cache_hls)
                   and os.path.exists(cache_s30)
                   and os.path.exists(cache_l30))
    if _all_cached:
        print('Loading cached results from disk...')
        for label, path in [('HLS', cache_hls), ('S30', cache_s30), ('L30', cache_l30)]:
            print(f'  {label}: {path}')
        df_hls = gpd.read_file(cache_hls)
        df_s30 = gpd.read_file(cache_s30)
        df_l30 = gpd.read_file(cache_l30)
        print(f'\n  HLS pixels loaded: {len(df_hls)}')
        print(f'  S30 pixels loaded: {len(df_s30)}')
        print(f'  L30 pixels loaded: {len(df_l30)}')

if not _all_cached:
    print('FORCE_RECOMPUTE=True — rerunning GEE.' if FORCE_RECOMPUTE
          else 'No cache found — running GEE computation...')

    print('\n  HLS combined:')
    ccd_hls_spatial = load_or_compute_ccdc(
        hls_col, SITE_CONFIG['region'], SITE_CONFIG,
        asset_path=SITE_CONFIG['ccdc_asset_hls']
    )
    print('  S30 only:')
    ccd_s30_spatial = load_or_compute_ccdc(
        s30_col, SITE_CONFIG['region'], SITE_CONFIG,
        asset_path=SITE_CONFIG.get('ccdc_asset_s30')   # optional asset key
    )
    print('  L30 only:')
    ccd_l30_spatial = load_or_compute_ccdc(
        l30_col, SITE_CONFIG['region'], SITE_CONFIG,
        asset_path=SITE_CONFIG['ccdc_asset_l30']
    )

    print(f'\nReducing CCDC array images → scalar ({map_band}) server-side...')
    scalar_hls = ccdc_array_to_scalar(ccd_hls_spatial, map_band)
    scalar_s30 = ccdc_array_to_scalar(ccd_s30_spatial, map_band)
    scalar_l30 = ccdc_array_to_scalar(ccd_l30_spatial, map_band)

    print(f'Sampling scalar images at {map_scale} m ...')
    df_hls = scalar_ccdc_to_gdf(scalar_hls, SITE_CONFIG['region'], scale=map_scale, tiles=map_tiles)
    df_s30 = scalar_ccdc_to_gdf(scalar_s30, SITE_CONFIG['region'], scale=map_scale, tiles=map_tiles)
    df_l30 = scalar_ccdc_to_gdf(scalar_l30, SITE_CONFIG['region'], scale=map_scale, tiles=map_tiles)

    print(f'\n  HLS pixels sampled: {len(df_hls)}')
    print(f'  S30 pixels sampled: {len(df_s30)}')
    print(f'  L30 pixels sampled: {len(df_l30)}')

    # Save all three to GeoJSON for reuse
    df_hls.to_file(cache_hls, driver='GeoJSON')
    df_s30.to_file(cache_s30, driver='GeoJSON')
    df_l30.to_file(cache_l30, driver='GeoJSON')
    print(f'\nResults cached to disk:')
    for path in [cache_hls, cache_s30, cache_l30]:
        print(f'  {path}')

---
## Figure 3 — Spatial Disturbance Maps
Compare the year of maximum CCDC change across `L30`, `S30`, and combined `HLS` using a shared color scale and common map extent.


In [ ]:
# Plot controls
# - 'data': zoom to the actual CCDC GeoDataFrame bounds. Best for avoiding blank
#   maps when cached CCDC results were generated from a different SITE_CONFIG.
# - 'site': force the current SITE_CONFIG region. Best when you are sure the
#   cached CCDC files match the selected site.
from gee_hls.notebook_helpers import (
    clean_disturbance_df,
    plot_disturbance_map,
    shared_disturbance_extent,
    site_extent_from_config,
)

DISTURBANCE_EXTENT_MODE = 'data'  # 'data' or 'site'

df_l30 = clean_disturbance_df(df_l30)
df_s30 = clean_disturbance_df(df_s30)
df_hls = clean_disturbance_df(df_hls)

yr_min = SITE_CONFIG['map_year_min']
yr_max = SITE_CONFIG['map_year_max']
site_extent = site_extent_from_config(SITE_CONFIG)

print('CCDC disturbance map diagnostics:')
print(
    f"  SITE_CONFIG region: [{site_extent[0]:.4f}, {site_extent[2]:.4f}, "
    f"{site_extent[1]:.4f}, {site_extent[3]:.4f}]"
)

sensor_df_lookup = {'L30': df_l30, 'S30': df_s30, 'HLS': df_hls}
shared_extent = shared_disturbance_extent(sensor_df_lookup, SITE_CONFIG, mode=DISTURBANCE_EXTENT_MODE)
print(
    f"  Plot extent ({DISTURBANCE_EXTENT_MODE}): "
    f"[{shared_extent[0]:.4f}, {shared_extent[2]:.4f}, {shared_extent[1]:.4f}, {shared_extent[3]:.4f}]"
)

fig3, axes = plt.subplots(1, 3, figsize=(21, 7), constrained_layout=True)

sensor_panels = [
    (df_l30, 'L30 Only\nLandsat 8/9'),
    (df_s30, 'S30 Only\nSentinel-2 A/B'),
    (df_hls, 'HLS Combined\nL30 + S30'),
]

for ax, (df_sensor, title) in zip(axes, sensor_panels):
    plot_disturbance_map(
        df_sensor,
        title=f'{title}\n{map_band} max-change year, {yr_min}–{yr_max}',
        year_min=yr_min,
        year_max=yr_max,
        extent=shared_extent,
        site_extent=site_extent,
        cmap='YlOrRd',
        ax=ax,
    )

sm = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=yr_min, vmax=yr_max), cmap='YlOrRd')
sm.set_array([])
cbar = fig3.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.78, pad=0.015)
cbar.set_label('Year of max change', fontsize=11)

fig3.suptitle(
    f'{SITE_CONFIG["site_name"]} — Spatial CCDC Disturbance Patterns by Sensor',
    fontsize=15,
    fontweight='bold',
)

fig3_path = os.path.join(SITE_CONFIG['output_dir'], f"{SITE_CONFIG['site_name']}_fig3_l30_s30_hls_disturbance_maps.png")
fig3.savefig(fig3_path, dpi=150, bbox_inches='tight')
print(f'Fig 3 saved → {fig3_path}')
plt.show()

---
## Step 4 — Hansen Reference Preview (Recommended)
Before running quantitative validation, inspect the Hansen baseline forest mask and recent loss layers over the exact CCDC domain.

This is the quickest way to catch an extent mismatch or an unexpected reference-data issue.


---
## Figure 4 — Hansen Validation: L30 vs S30 vs HLS
Compare binary spatial CCDC disturbance detections against Hansen Global Forest Change using a common validation grid and zonal aggregation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HANSEN REFERENCE VISUALIZATION — same domain as CCDC validation
#
# This cell helps sanity-check the reference data before validation. It plots:
#   1. Hansen treecover2000 (% canopy cover)
#   2. Baseline forest mask used for validation
#   3. Hansen loss year within the HLS-era validation window
# ═══════════════════════════════════════════════════════════════════════════════

import io
import math
from matplotlib.colors import ListedColormap

HANSEN_VIZ_TREECOVER_MIN = 30
HANSEN_VIZ_START_YEAR = max(SITE_CONFIG['start_year'], 2015)
HANSEN_VIZ_END_YEAR = min(SITE_CONFIG['end_year'], 2024)
HANSEN_VIZ_SCALE = max(map_scale, 300)  # metres; increase to 500/1000 for faster preview

if HANSEN_VIZ_START_YEAR > HANSEN_VIZ_END_YEAR:
    raise ValueError('Hansen visualization requires overlap with the 2015–2024 Hansen loss window.')


def hansen_viz_region_bounds(region):
    coords = region.bounds().getInfo()['coordinates'][0]
    xs = [pt[0] for pt in coords]
    ys = [pt[1] for pt in coords]
    return min(xs), min(ys), max(xs), max(ys)


def hansen_viz_grid(region, scale_m):
    x_min, y_min, x_max, y_max = hansen_viz_region_bounds(region)
    lat_c = (y_min + y_max) / 2.0
    dy = scale_m / 111_320.0
    dx = scale_m / (111_320.0 * max(math.cos(math.radians(lat_c)), 1e-6))
    width = max(1, round((x_max - x_min) / dx))
    height = max(1, round((y_max - y_min) / dy))
    return {
        'x_min': x_min,
        'x_max': x_max,
        'y_min': y_min,
        'y_max': y_max,
        'dx': dx,
        'dy': dy,
        'width': width,
        'height': height,
        'extent': [x_min, x_max, y_min, y_max],
    }


def download_hansen_viz_layers(region, scale_m):
    grid = hansen_viz_grid(region, scale_m)

    gfc_viz = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
    treecover = gfc_viz.select('treecover2000')
    lossyear = gfc_viz.select('lossyear')

    forest_mask = treecover.gte(HANSEN_VIZ_TREECOVER_MIN).rename('forest_mask').toByte()
    loss_mask = (
        lossyear.gte(HANSEN_VIZ_START_YEAR - 2000)
        .And(lossyear.lte(HANSEN_VIZ_END_YEAR - 2000))
        .And(forest_mask)
        .rename('loss_mask')
        .toByte()
    )
    loss_year = (
        lossyear.add(2000)
        .updateMask(loss_mask)
        .rename('loss_year')
        .toInt16()
    )

    viz_img = (
        treecover.rename('treecover2000')
        .addBands(forest_mask)
        .addBands(loss_mask)
        .addBands(loss_year.unmask(0))
        .clip(region)
    )

    raw = ee.data.computePixels({
        'expression': viz_img,
        'fileFormat': 'NPY',
        'grid': {
            'crsCode': 'EPSG:4326',
            'affineTransform': {
                'scaleX': grid['dx'],
                'shearX': 0.0,
                'translateX': grid['x_min'],
                'shearY': 0.0,
                'scaleY': -grid['dy'],
                'translateY': grid['y_max'],
            },
            'dimensions': {'width': grid['width'], 'height': grid['height']},
        },
    })

    return np.load(io.BytesIO(raw)), grid


print(
    f'Downloading Hansen reference preview for {SITE_CONFIG["site_name"]} '
    f'({HANSEN_VIZ_START_YEAR}–{HANSEN_VIZ_END_YEAR}) at {HANSEN_VIZ_SCALE} m ...'
)
hansen_viz_arr, hansen_viz_grid_info = download_hansen_viz_layers(
    SITE_CONFIG['region'],
    HANSEN_VIZ_SCALE,
)

_tree = hansen_viz_arr['treecover2000'].astype(float)
_forest = hansen_viz_arr['forest_mask'].astype(bool)
_loss = hansen_viz_arr['loss_mask'].astype(bool)
_loss_year = hansen_viz_arr['loss_year'].astype(float)
_loss_year[_loss_year == 0] = np.nan

print(
    f"  Grid: {hansen_viz_grid_info['width']} x {hansen_viz_grid_info['height']} pixels; "
    f"forest pixels={int(_forest.sum()):,}; "
    f"loss pixels={int(_loss.sum()):,}; "
    f"loss fraction={100 * _loss.sum() / max(_forest.sum(), 1):.1f}%"
)

fig_hansen_ref, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
extent = hansen_viz_grid_info['extent']

im0 = axes[0].imshow(
    _tree,
    extent=extent,
    origin='upper',
    cmap='Greens',
    vmin=0,
    vmax=100,
)
axes[0].set_title('Hansen treecover2000 (%)', fontweight='bold')
cb0 = fig_hansen_ref.colorbar(im0, ax=axes[0], shrink=0.75)
cb0.set_label('Canopy cover (%)')

forest_display = np.where(_forest, 1, np.nan)
im1 = axes[1].imshow(
    forest_display,
    extent=extent,
    origin='upper',
    cmap=ListedColormap(['#2E7D32']),
    vmin=0,
    vmax=1,
)
axes[1].set_title(f'Validation forest mask\ntreecover2000 >= {HANSEN_VIZ_TREECOVER_MIN}%', fontweight='bold')

im2 = axes[2].imshow(
    _loss_year,
    extent=extent,
    origin='upper',
    cmap='YlOrRd',
    vmin=HANSEN_VIZ_START_YEAR,
    vmax=HANSEN_VIZ_END_YEAR,
)
axes[2].set_title(f'Hansen loss year\n{HANSEN_VIZ_START_YEAR}–{HANSEN_VIZ_END_YEAR}', fontweight='bold')
cb2 = fig_hansen_ref.colorbar(im2, ax=axes[2], shrink=0.75)
cb2.set_label('Loss year')

for ax in axes:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(False)
    ax.tick_params(labelsize=8)

fig_hansen_ref.suptitle(
    f'{SITE_CONFIG["site_name"]} — Hansen Reference Layers for CCDC Validation',
    fontsize=14,
    fontweight='bold',
)

hansen_ref_path = os.path.join(
    SITE_CONFIG['output_dir'],
    f"{SITE_CONFIG['site_name']}_hansen_reference_preview.png",
)
fig_hansen_ref.savefig(hansen_ref_path, dpi=150, bbox_inches='tight')
print(f'Hansen reference preview saved → {hansen_ref_path}')
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HANSEN VALIDATION — local zonal mean on a CCDC-scale grid
#
# Workflow:
#   1. Download Hansen forest/loss masks locally at a fine resolution.
#   2. Aggregate Hansen to CCDC-scale validation cells using zonal statistics.
#   3. Rasterize local CCDC detections to the same validation grid.
#   4. Compare L30, S30, and HLS predictions against Hansen labels.
#
# This avoids Earth Engine memory errors and avoids comparing offset grids by
# centroid only. The validation unit is a regular grid at CCDC_VALIDATION_SCALE.
# ═══════════════════════════════════════════════════════════════════════════════

import glob
import io
import math

HANSEN_VALIDATION_TREECOVER_MIN = 30
HANSEN_VALIDATION_START_YEAR = max(SITE_CONFIG['start_year'], 2015)
HANSEN_VALIDATION_END_YEAR = min(SITE_CONFIG['end_year'], 2024)

# Hansen and CCDC are evaluated on the same CCDC-scale grid.
# Set HANSEN_FINE_SCALE lower than CCDC_VALIDATION_SCALE only if you explicitly
# want sub-cell zonal fractions.
CCDC_VALIDATION_SCALE = map_scale
HANSEN_FINE_SCALE = CCDC_VALIDATION_SCALE

# A CCDC-scale cell is considered reference forest if at least this fraction of
# fine Hansen pixels are baseline forest. It is considered reference loss if at
# least this fraction of its forest pixels are Hansen loss during the window.
HANSEN_MIN_FOREST_FRAC = 0.25
HANSEN_LOSS_FRAC_THRESHOLD = 0.10

if HANSEN_VALIDATION_START_YEAR > HANSEN_VALIDATION_END_YEAR:
    raise ValueError('Hansen validation requires overlap with the 2015–2024 Hansen loss window.')


def region_bounds_lonlat(region):
    """Return lon/lat bounds from an ee.Geometry.Rectangle-like region."""
    coords = region.bounds().getInfo()['coordinates'][0]
    xs = [pt[0] for pt in coords]
    ys = [pt[1] for pt in coords]
    return min(xs), min(ys), max(xs), max(ys)


def make_lonlat_grid(region, scale_m):
    """Create an affine lon/lat grid covering region at approximately scale_m."""
    x_min, y_min, x_max, y_max = region_bounds_lonlat(region)
    lat_c = (y_min + y_max) / 2.0
    dy = scale_m / 111_320.0
    dx = scale_m / (111_320.0 * max(math.cos(math.radians(lat_c)), 1e-6))
    width = max(1, int(math.ceil((x_max - x_min) / dx)))
    height = max(1, int(math.ceil((y_max - y_min) / dy)))
    return {
        'x_min': x_min,
        'x_max': x_min + width * dx,
        'y_min': y_max - height * dy,
        'y_max': y_max,
        'dx': dx,
        'dy': dy,
        'width': width,
        'height': height,
        'extent': [x_min, x_min + width * dx, y_max - height * dy, y_max],
    }



def ccdc_union_extent(sensor_dfs, pad_frac=0.0):
    """Get validation extent directly from loaded CCDC GeoDataFrame bounds."""
    bounds_list = []
    for label, df in sensor_dfs:
        if df is None or df.empty or 'geometry' not in df.columns:
            print(f'  {label}: no geometry available for CCDC extent.')
            continue

        work = df
        try:
            if getattr(work, 'crs', None) is not None and str(work.crs).upper() not in ('EPSG:4326', 'WGS84'):
                work = work.to_crs('EPSG:4326')
            bounds = work.total_bounds  # minx, miny, maxx, maxy
        except Exception as exc:
            print(f'  {label}: could not read bounds ({exc})')
            continue

        if len(bounds) == 4 and np.isfinite(bounds).all():
            bounds_list.append(bounds)
            print(
                f'  {label} CCDC extent: '
                f'[{bounds[0]:.4f}, {bounds[1]:.4f}, {bounds[2]:.4f}, {bounds[3]:.4f}]'
            )

    if not bounds_list:
        raise ValueError(
            'No valid CCDC GeoDataFrame bounds found. Run Step 3 or load matching cache files before validation.'
        )

    bounds_arr = np.vstack(bounds_list)
    x_min = float(bounds_arr[:, 0].min())
    y_min = float(bounds_arr[:, 1].min())
    x_max = float(bounds_arr[:, 2].max())
    y_max = float(bounds_arr[:, 3].max())

    dx = x_max - x_min
    dy = y_max - y_min
    x_pad = dx * pad_frac
    y_pad = dy * pad_frac
    return [x_min - x_pad, y_min - y_pad, x_max + x_pad, y_max + y_pad]


def rectangle_from_extent(extent):
    """Build an ee.Geometry.Rectangle from [xmin, ymin, xmax, ymax]."""
    return ee.Geometry.Rectangle(extent, geodesic=False)


def download_hansen_fine_layers(region, scale_m):
    """Download fine Hansen baseline-forest, recent-loss, and loss-year layers."""
    grid = make_lonlat_grid(region, scale_m)

    gfc = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')
    treecover = gfc.select('treecover2000')
    lossyear = gfc.select('lossyear')

    forest_mask = treecover.gte(HANSEN_VALIDATION_TREECOVER_MIN).rename('forest_mask').toByte()
    loss_mask = (
        lossyear.gte(HANSEN_VALIDATION_START_YEAR - 2000)
        .And(lossyear.lte(HANSEN_VALIDATION_END_YEAR - 2000))
        .And(forest_mask)
        .rename('loss_mask')
        .toByte()
    )
    loss_year = (
        lossyear.add(2000)
        .updateMask(loss_mask)
        .rename('loss_year')
        .toInt16()
    )

    img = forest_mask.addBands(loss_mask).addBands(loss_year.unmask(0)).unmask(0).clip(region)
    raw = ee.data.computePixels({
        'expression': img,
        'fileFormat': 'NPY',
        'grid': {
            'crsCode': 'EPSG:4326',
            'affineTransform': {
                'scaleX': grid['dx'],
                'shearX': 0.0,
                'translateX': grid['x_min'],
                'shearY': 0.0,
                'scaleY': -grid['dy'],
                'translateY': grid['y_max'],
            },
            'dimensions': {'width': grid['width'], 'height': grid['height']},
        },
    })

    arr = np.load(io.BytesIO(raw))
    return (
        arr['forest_mask'].astype(np.uint8),
        arr['loss_mask'].astype(np.uint8),
        arr['loss_year'].astype(np.int16),
        grid,
    )


def aggregate_hansen_to_ccdc_grid(forest_fine, loss_fine, loss_year_fine, fine_grid, ccdc_grid):
    """Aggregate fine Hansen pixels to CCDC-scale cells using zonal statistics."""
    h_f, w_f = forest_fine.shape
    x_centers = fine_grid['x_min'] + fine_grid['dx'] * (np.arange(w_f) + 0.5)
    y_centers = fine_grid['y_max'] - fine_grid['dy'] * (np.arange(h_f) + 0.5)

    fine_cols = np.floor((x_centers - ccdc_grid['x_min']) / ccdc_grid['dx']).astype(int)
    fine_rows = np.floor((ccdc_grid['y_max'] - y_centers) / ccdc_grid['dy']).astype(int)

    col_grid, row_grid = np.meshgrid(fine_cols, fine_rows)
    valid = (
        (col_grid >= 0) & (col_grid < ccdc_grid['width'])
        & (row_grid >= 0) & (row_grid < ccdc_grid['height'])
    )

    shape = (ccdc_grid['height'], ccdc_grid['width'])
    fine_count = np.zeros(shape, dtype=np.float32)
    forest_sum = np.zeros(shape, dtype=np.float32)
    loss_sum = np.zeros(shape, dtype=np.float32)
    max_loss_year = np.zeros(shape, dtype=np.int16)

    rows = row_grid[valid]
    cols = col_grid[valid]
    np.add.at(fine_count, (rows, cols), 1)
    np.add.at(forest_sum, (rows, cols), forest_fine[valid])
    np.add.at(loss_sum, (rows, cols), loss_fine[valid])
    np.maximum.at(max_loss_year, (rows, cols), loss_year_fine[valid])

    forest_frac = np.divide(
        forest_sum, fine_count,
        out=np.zeros_like(forest_sum, dtype=np.float32),
        where=fine_count > 0,
    )
    loss_frac = np.divide(
        loss_sum, forest_sum,
        out=np.zeros_like(loss_sum, dtype=np.float32),
        where=forest_sum > 0,
    )

    eval_mask = forest_frac >= HANSEN_MIN_FOREST_FRAC
    reference_loss = (loss_frac >= HANSEN_LOSS_FRAC_THRESHOLD) & eval_mask
    max_loss_year = np.where(reference_loss, max_loss_year, 0)
    return forest_frac, loss_frac, max_loss_year, eval_mask, reference_loss


def cache_candidates(sensor_key):
    """Find likely local cache files for the current site/sensor."""
    candidates = []
    exact_var = globals().get(f'cache_{sensor_key}')
    if exact_var:
        candidates.append(exact_var)

    search_dirs = []
    if '_cache_dir' in globals():
        search_dirs.append(_cache_dir)
    search_dirs.append(SITE_CONFIG['output_dir'])
    search_dirs = list(dict.fromkeys(search_dirs))

    tag = globals().get('_cache_tag')
    for search_dir in search_dirs:
        if tag:
            candidates.extend(glob.glob(os.path.join(search_dir, f'cache_{sensor_key}_{tag}.geojson')))
        candidates.extend(glob.glob(os.path.join(
            search_dir,
            f"cache_{sensor_key}_{SITE_CONFIG['site_name']}_{map_band}_*.geojson",
        )))

    return [path for path in dict.fromkeys(candidates) if path and os.path.exists(path)]


def prepare_ccdc_centroids(df, grid, start_year, end_year):
    """Return CCDC centroid row/col indices plus diagnostics."""
    diagnostics = {
        'n_total': 0,
        'n_year_valid': 0,
        'n_in_window': 0,
        'n_inside_grid': 0,
        'n_predicted_cells': 0,
        'bounds': None,
        'message': None,
    }

    if df is None or df.empty:
        diagnostics['message'] = 'GeoDataFrame is empty or not loaded.'
        return None, None, diagnostics

    diagnostics['n_total'] = len(df)
    if 'year_of_max_change' not in df.columns:
        diagnostics['message'] = 'Missing year_of_max_change column.'
        return None, None, diagnostics

    work = df.copy()
    work['year_of_max_change'] = pd.to_numeric(work['year_of_max_change'], errors='coerce')
    work = work.dropna(subset=['year_of_max_change'])
    diagnostics['n_year_valid'] = len(work)

    work = work[
        (work['year_of_max_change'] >= start_year)
        & (work['year_of_max_change'] <= end_year)
    ].copy()
    diagnostics['n_in_window'] = len(work)
    if work.empty:
        diagnostics['message'] = 'No CCDC detections inside validation year window.'
        return None, None, diagnostics

    if getattr(work, 'crs', None) is not None and str(work.crs).upper() not in ('EPSG:4326', 'WGS84'):
        work = work.to_crs('EPSG:4326')

    try:
        diagnostics['bounds'] = [float(v) for v in work.total_bounds]
    except Exception:
        pass

    # CCDC GeoDataFrames represent changed cells. Mapping centroids onto the
    # CCDC-scale validation grid marks those cells as predicted disturbance.
    centroids = work.geometry.centroid
    xs = np.array([geom.x for geom in centroids])
    ys = np.array([geom.y for geom in centroids])

    cols = np.floor((xs - grid['x_min']) / grid['dx']).astype(int)
    rows = np.floor((grid['y_max'] - ys) / grid['dy']).astype(int)
    inside = (
        (cols >= 0) & (cols < grid['width'])
        & (rows >= 0) & (rows < grid['height'])
    )
    diagnostics['n_inside_grid'] = int(inside.sum())
    if diagnostics['n_inside_grid'] == 0:
        diagnostics['message'] = 'CCDC detections are outside the CCDC validation grid.'
        return None, None, diagnostics

    rows = rows[inside]
    cols = cols[inside]
    diagnostics['n_predicted_cells'] = len(set(zip(rows.tolist(), cols.tolist())))
    return rows, cols, diagnostics


def load_better_cache_if_needed(sensor_key, sensor_label, df, grid):
    """If current df has no grid overlap, try matching cache files."""
    _, _, current_diag = prepare_ccdc_centroids(
        df,
        grid,
        HANSEN_VALIDATION_START_YEAR,
        HANSEN_VALIDATION_END_YEAR,
    )
    if current_diag['n_inside_grid'] > 0:
        return df, current_diag, None

    for path in cache_candidates(sensor_key):
        try:
            cached = gpd.read_file(path)
        except Exception as exc:
            print(f'    Could not load cache {path}: {exc}')
            continue

        _, _, cached_diag = prepare_ccdc_centroids(
            cached,
            grid,
            HANSEN_VALIDATION_START_YEAR,
            HANSEN_VALIDATION_END_YEAR,
        )
        if cached_diag['n_inside_grid'] > current_diag['n_inside_grid']:
            print(f'    Using cache for {sensor_label}: {path}')
            return cached, cached_diag, path

    return df, current_diag, None


def ccdc_prediction_grid(df, grid, sensor_label):
    """Rasterize CCDC detections to the CCDC-scale validation grid."""
    pred = np.zeros((grid['height'], grid['width']), dtype=bool)
    rows, cols, diagnostics = prepare_ccdc_centroids(
        df,
        grid,
        HANSEN_VALIDATION_START_YEAR,
        HANSEN_VALIDATION_END_YEAR,
    )

    print(
        f"  {sensor_label} CCDC diagnostics: total={diagnostics['n_total']:,}, "
        f"valid_year={diagnostics['n_year_valid']:,}, "
        f"in_window={diagnostics['n_in_window']:,}, "
        f"inside_grid={diagnostics['n_inside_grid']:,}, "
        f"predicted_cells={diagnostics['n_predicted_cells']:,}"
    )
    if diagnostics['bounds'] is not None:
        b = diagnostics['bounds']
        print(f'    {sensor_label} bounds: [{b[0]:.4f}, {b[1]:.4f}, {b[2]:.4f}, {b[3]:.4f}]')
    if diagnostics['message']:
        print(f"    WARNING: {sensor_label}: {diagnostics['message']}")

    if rows is not None and cols is not None:
        pred[rows, cols] = True
    return pred, diagnostics


def metrics_from_binary_arrays(reference, prediction, eval_mask):
    """Calculate binary validation metrics over eligible forest cells."""
    ref = reference[eval_mask].astype(bool)
    pred = prediction[eval_mask].astype(bool)

    tp = int(np.logical_and(ref, pred).sum())
    tn = int(np.logical_and(~ref, ~pred).sum())
    fp = int(np.logical_and(~ref, pred).sum())
    fn = int(np.logical_and(ref, ~pred).sum())
    total = tp + tn + fp + fn

    accuracy = (tp + tn) / total if total else np.nan
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    balanced_accuracy = np.nanmean([recall, specificity])

    return {
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'iou': iou,
        'n_eval_cells': total,
        'n_hansen_loss_cells': int(ref.sum()),
        'n_ccdc_loss_cells': int(pred.sum()),
    }


print(
    'Validating CCDC disturbance maps with local Hansen zonal means '
    f'({HANSEN_VALIDATION_START_YEAR}–{HANSEN_VALIDATION_END_YEAR}) ...'
)
print(f'  Hansen fine scale: {HANSEN_FINE_SCALE} m')
print(f'  CCDC validation grid scale: {CCDC_VALIDATION_SCALE} m')
print(f'  Forest cell threshold: forest_frac >= {HANSEN_MIN_FOREST_FRAC:.2f}')
print(f'  Hansen loss threshold: loss_frac_of_forest >= {HANSEN_LOSS_FRAC_THRESHOLD:.2f}')

sensor_extent_inputs = [
    ('L30', df_l30),
    ('S30', df_s30),
    ('HLS', df_hls),
]
ccdc_extent = ccdc_union_extent(sensor_extent_inputs, pad_frac=0.0)
hansen_validation_region = rectangle_from_extent(ccdc_extent)
print(
    f'  Hansen/validation domain from loaded CCDC GeoJSON extent: '
    f'[{ccdc_extent[0]:.4f}, {ccdc_extent[1]:.4f}, {ccdc_extent[2]:.4f}, {ccdc_extent[3]:.4f}]'
)

ccdc_grid = make_lonlat_grid(hansen_validation_region, CCDC_VALIDATION_SCALE)
forest_fine, loss_fine, loss_year_fine, fine_grid = download_hansen_fine_layers(
    hansen_validation_region,
    HANSEN_FINE_SCALE,
)
forest_frac, loss_frac, max_loss_year, eval_mask, reference_loss = aggregate_hansen_to_ccdc_grid(
    forest_fine,
    loss_fine,
    loss_year_fine,
    fine_grid,
    ccdc_grid,
)

print(
    f"  Fine Hansen grid: {fine_grid['width']} x {fine_grid['height']} pixels; "
    f"fine forest px={int(forest_fine.sum()):,}; fine loss px={int(loss_fine.sum()):,}"
)
print(
    f"  CCDC validation grid: {ccdc_grid['width']} x {ccdc_grid['height']} cells; "
    f"eligible forest cells={int(eval_mask.sum()):,}; "
    f"Hansen loss cells={int(reference_loss[eval_mask].sum()):,}"
)

sensor_inputs = [
    ('L30', 'l30', df_l30),
    ('S30', 's30', df_s30),
    ('HLS', 'hls', df_hls),
]

validation_rows = []
prediction_maps = {}
for sensor_label, sensor_key, df_sensor in sensor_inputs:
    df_sensor, _, cache_path = load_better_cache_if_needed(sensor_key, sensor_label, df_sensor, ccdc_grid)
    if sensor_key == 'l30':
        df_l30 = df_sensor
    elif sensor_key == 's30':
        df_s30 = df_sensor
    elif sensor_key == 'hls':
        df_hls = df_sensor

    prediction, diagnostics = ccdc_prediction_grid(df_sensor, ccdc_grid, sensor_label)
    prediction_maps[sensor_label] = prediction
    metrics = metrics_from_binary_arrays(reference_loss, prediction, eval_mask)
    metrics['sensor'] = sensor_label
    metrics['cache_path'] = cache_path
    validation_rows.append(metrics)

    print(
        f"  {sensor_label}: F1={metrics['f1']:.3f}, "
        f"Balanced Acc={metrics['balanced_accuracy']:.3f}, "
        f"Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}, "
        f"CCDC loss cells={metrics['n_ccdc_loss_cells']:,}"
    )

df_hansen_validation = pd.DataFrame(validation_rows).set_index('sensor')

print('\nHansen zonal validation metrics (%):')
display_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'iou']
print(
    (df_hansen_validation[display_metrics] * 100)
    .round(1)
    .to_string()
)

if (df_hansen_validation['n_ccdc_loss_cells'] == 0).all():
    print(
        '\nWARNING: all CCDC prediction grids are empty. This is not a valid accuracy comparison.\n'
        'Run Step 3 for the current SITE_CONFIG, verify Fig 3 shows changed pixels inside the dashed site box,\n'
        'or set CACHE_DIR to the folder containing matching cache_l30/cache_s30/cache_hls GeoJSON files.'
    )
else:
    best_f1_sensor = df_hansen_validation['f1'].fillna(-1).idxmax()
    best_bal_sensor = df_hansen_validation['balanced_accuracy'].fillna(-1).idxmax()
    if best_f1_sensor == 'HLS' or best_bal_sensor == 'HLS':
        print('\nResult: this site supports the expected HLS advantage against Hansen.')
    else:
        print('\nResult: HLS is not the top-scoring sensor for this sample; inspect maps/cache/site settings.')

# Plot validation metrics and the Hansen year-of-loss reference used for comparison.
plot_cols = ['balanced_accuracy', 'precision', 'recall', 'f1', 'iou']
fig4, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

(df_hansen_validation[plot_cols] * 100).plot(
    kind='bar',
    ax=axes[0],
    color=['#455A64', '#1565C0', '#2E7D32', '#EF6C00', '#6A1B9A'],
    width=0.78,
)
axes[0].set_ylim(0, 100)
axes[0].set_ylabel('Metric value (%)')
axes[0].set_xlabel('Sensor configuration')
axes[0].set_title('Validation Metrics', fontweight='bold')
axes[0].grid(axis='y', alpha=0.25)
axes[0].legend(title='Metric', fontsize=8)

year_display = np.where(max_loss_year > 0, max_loss_year, np.nan)
im = axes[1].imshow(
    year_display,
    extent=ccdc_grid['extent'],
    origin='upper',
    cmap='YlOrRd',
    vmin=HANSEN_VALIDATION_START_YEAR,
    vmax=HANSEN_VALIDATION_END_YEAR,
)
axes[1].set_title('Hansen Max Loss Year\naggregated to CCDC-scale cells', fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].grid(False)
cb = fig4.colorbar(im, ax=axes[1], shrink=0.8)
cb.set_label('Max Hansen loss year')

fig4.suptitle(
    f'Hansen Zonal Validation of CCDC Disturbance Maps\n'
    f'{SITE_CONFIG["site_name"]}, {HANSEN_VALIDATION_START_YEAR}–{HANSEN_VALIDATION_END_YEAR}',
    fontsize=13,
    fontweight='bold',
)

fig4_path = os.path.join(
    SITE_CONFIG['output_dir'],
    f"{SITE_CONFIG['site_name']}_fig4_hansen_zonal_year_validation_l30_s30_hls.png",
)
fig4.savefig(fig4_path, dpi=150, bbox_inches='tight')
print(f'Fig 4 saved → {fig4_path}')
plt.show()


---
## Sensor Diagnostics
Use these diagnostics to interpret *why* validation performance differs across `L30`, `S30`, and `HLS`.

The cell checks:
- temporal variability at the study pixel
- spatial fragmentation of the predicted disturbance maps


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SENSOR DIAGNOSTICS — explore why sensor performance differs
#
# Uses two lightweight diagnostics already available in the notebook:
#   1. Pixel-level time-series variability at the study pixel
#   2. Spatial fragmentation of the binary disturbance maps used in validation
#
# These are not definitive proof, but they help test whether a sensor is
# denser yet noisier and/or more spatially fragmented.
# ═══════════════════════════════════════════════════════════════════════════════

from gee_hls.notebook_helpers import fragmentation_stats, robust_ts_stats

if 'df_hansen_validation' not in globals() or 'prediction_maps' not in globals():
    raise RuntimeError('Run the Hansen validation cell before running sensor diagnostics.')

if 'map_band' not in globals():
    raise RuntimeError('map_band is not defined. Run the spatial CCDC cells first.')

sensor_ts = {
    'L30': globals().get('ts_l30'),
    'S30': globals().get('ts_s30'),
    'HLS': globals().get('ts_hls'),
}
sensor_preds = {
    'L30': prediction_maps.get('L30'),
    'S30': prediction_maps.get('S30'),
    'HLS': prediction_maps.get('HLS'),
}

var_rows = []
for sensor in ['L30', 'S30', 'HLS']:
    ts_stats = robust_ts_stats(sensor_ts.get(sensor), map_band)
    frag_stats = fragmentation_stats(sensor_preds.get(sensor))

    row = {'sensor': sensor}
    row.update(ts_stats)
    row.update(frag_stats)
    if sensor in df_hansen_validation.index:
        for metric in ['accuracy', 'precision', 'recall', 'f1', 'iou']:
            row[metric] = float(df_hansen_validation.loc[sensor, metric])
    var_rows.append(row)

sensor_diag = pd.DataFrame(var_rows).set_index('sensor')

display_cols = [
    'n_obs', 'median_gap_days', 'value_mad', 'diff_mad', 'yearly_count_cv',
    'predicted_cells', 'n_patches', 'median_patch_cells', 'singleton_share',
    'precision', 'recall', 'f1', 'iou'
]

print('Sensor diagnostics summary:')
print(sensor_diag[display_cols].round(3).to_string())

if {'L30', 'S30'}.issubset(sensor_diag.index):
    s30_noisier = sensor_diag.loc['S30', 'diff_mad'] > sensor_diag.loc['L30', 'diff_mad']
    s30_more_fragmented = sensor_diag.loc['S30', 'singleton_share'] > sensor_diag.loc['L30', 'singleton_share']
    s30_more_patches = sensor_diag.loc['S30', 'n_patches'] > sensor_diag.loc['L30', 'n_patches']

    print('\nDiagnostic reading:')
    if s30_noisier:
        print('- S30 shows higher short-term variability at the study pixel than L30, consistent with a noisier input signal.')
    else:
        print('- S30 does not show higher short-term variability than L30 at the study pixel in this diagnostic.')

    if s30_more_fragmented or s30_more_patches:
        print('- S30 disturbance predictions are more spatially fragmented than L30, which is consistent with noisier or less coherent detections.')
    else:
        print('- S30 disturbance predictions are not more fragmented than L30 in this diagnostic.')

    if 'HLS' in sensor_diag.index:
        if sensor_diag.loc['HLS', 'recall'] >= max(sensor_diag.loc['L30', 'recall'], sensor_diag.loc['S30', 'recall']):
            print('- HLS retains the strongest sensitivity (highest recall), supporting the idea that combining sensors improves disturbance detection completeness.')
        if sensor_diag.loc['HLS', 'singleton_share'] <= sensor_diag.loc['S30', 'singleton_share']:
            print('- HLS is at least as spatially coherent as S30 in terms of isolated 1-cell patches, suggesting the combined product reduces some S30-only noise.')

fig_diag, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
order = ['L30', 'S30', 'HLS']
colors = {'L30': '#1565C0', 'S30': '#2E7D32', 'HLS': '#EF6C00'}

ax = axes[0]
x = np.arange(len(order))
width = 0.36
ax.bar(x - width/2, sensor_diag.loc[order, 'diff_mad'], width,
       color=[colors[s] for s in order], alpha=0.9, label=f'{map_band} diff MAD')
ax.set_ylabel(f'{map_band} short-term variability')
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_title('Temporal Variability at Study Pixel', fontweight='bold')
ax.grid(axis='y', alpha=0.25)

ax2 = ax.twinx()
ax2.plot(x, sensor_diag.loc[order, 'n_obs'], color='#455A64', marker='o', linewidth=2.0, label='Observation count')
ax2.set_ylabel('Observation count')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

ax = axes[1]
ax.bar(x - width/2, sensor_diag.loc[order, 'singleton_share'] * 100, width,
       color=[colors[s] for s in order], alpha=0.9, label='Singleton patch share (%)')
ax.set_ylabel('Singleton patch share (%)')
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_title('Spatial Fragmentation vs Validation', fontweight='bold')
ax.grid(axis='y', alpha=0.25)

ax2 = ax.twinx()
ax2.plot(x, sensor_diag.loc[order, 'f1'] * 100, color='#8E24AA', marker='o', linewidth=2.0, label='F1 (%)')
ax2.plot(x, sensor_diag.loc[order, 'recall'] * 100, color='#E53935', marker='s', linewidth=2.0, label='Recall (%)')
ax2.set_ylabel('Validation metric (%)')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

fig_diag.suptitle(
    f'{SITE_CONFIG["site_name"]} — Sensor Diagnostics for Explaining Validation Differences',
    fontsize=14,
    fontweight='bold',
)

diag_path = os.path.join(SITE_CONFIG['output_dir'], f"{SITE_CONFIG['site_name']}_sensor_diagnostics.png")
fig_diag.savefig(diag_path, dpi=150, bbox_inches='tight')
print(f'\nSensor diagnostics figure saved → {diag_path}')


---
## Hansen Validation Report
Write a short Markdown report summarizing the validation setup, metric table, and main interpretation for the selected site.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HANSEN VALIDATION REPORT — write brief Markdown summary
#
# Requires df_hansen_validation from the Hansen zonal validation cell above.
# ═══════════════════════════════════════════════════════════════════════════════

if 'df_hansen_validation' not in globals():
    raise RuntimeError('Run the Hansen zonal validation cell before writing the report.')

required_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'iou']
missing_metrics = [m for m in required_metrics if m not in df_hansen_validation.columns]
if missing_metrics:
    raise ValueError(f'df_hansen_validation is missing required metrics: {missing_metrics}')

metrics_pct = (df_hansen_validation[required_metrics] * 100).round(1)
metrics_pct = metrics_pct.rename(columns={
    'accuracy': 'Accuracy (%)',
    'balanced_accuracy': 'Balanced accuracy (%)',
    'precision': 'Precision (%)',
    'recall': 'Recall (%)',
    'f1': 'F1 (%)',
    'iou': 'IoU (%)',
})


from gee_hls.notebook_helpers import markdown_table

metric_labels = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'F1 score',
    'iou': 'IoU',
}

best = {
    metric: df_hansen_validation[metric].fillna(-1).idxmax()
    for metric in required_metrics
}

best_values = {
    metric: 100 * df_hansen_validation.loc[sensor, metric]
    for metric, sensor in best.items()
}

hls_recall = 100 * df_hansen_validation.loc['HLS', 'recall'] if 'HLS' in df_hansen_validation.index else np.nan
l30_f1 = 100 * df_hansen_validation.loc['L30', 'f1'] if 'L30' in df_hansen_validation.index else np.nan
hls_f1 = 100 * df_hansen_validation.loc['HLS', 'f1'] if 'HLS' in df_hansen_validation.index else np.nan
s30_f1 = 100 * df_hansen_validation.loc['S30', 'f1'] if 'S30' in df_hansen_validation.index else np.nan

if best['f1'] == 'HLS':
    overall_sentence = (
        'HLS combined produced the strongest overall agreement with Hansen based on F1 score, '
        'supporting the expected benefit of combining L30 and S30 observations.'
    )
elif best['f1'] == 'L30':
    overall_sentence = (
        'L30 produced the strongest overall agreement with Hansen based on F1 score, '
        'indicating a more conservative map that aligns well with the Hansen reference in this test.'
    )
else:
    overall_sentence = (
        f'{best["f1"]} produced the strongest overall agreement with Hansen based on F1 score.'
    )

if best['recall'] == 'HLS':
    recall_sentence = (
        f'HLS had the highest recall ({hls_recall:.1f}%), meaning it detected the largest share of Hansen loss cells '
        'and had the lowest omission error among the three sensor configurations.'
    )
else:
    recall_sentence = (
        f'{best["recall"]} had the highest recall ({best_values["recall"]:.1f}%), meaning it detected the largest share of Hansen loss cells.'
    )

precision_sentence = (
    f'{best["precision"]} had the highest precision ({best_values["precision"]:.1f}%), '
    'meaning its detected disturbance cells most often corresponded to Hansen loss.'
)

iou_sentence = (
    f'{best["iou"]} had the highest IoU ({best_values["iou"]:.1f}%), '
    'indicating the strongest spatial overlap with Hansen loss cells.'
)

if 'L30' in df_hansen_validation.index and 'HLS' in df_hansen_validation.index:
    if hls_f1 < l30_f1:
        tradeoff_sentence = (
            f'HLS had slightly lower F1 than L30 ({hls_f1:.1f}% vs {l30_f1:.1f}%), '
            'suggesting a sensitivity-versus-precision tradeoff: HLS detects more loss, but also includes more cells not labeled as loss by Hansen.'
        )
    else:
        tradeoff_sentence = (
            f'HLS had F1 comparable to or higher than L30 ({hls_f1:.1f}% vs {l30_f1:.1f}%), '
            'suggesting that the combined time series improves detection without a major overall accuracy penalty.'
        )
else:
    tradeoff_sentence = 'The relative L30-HLS tradeoff could not be evaluated because one of the sensors is missing from the validation table.'

if 'S30' in df_hansen_validation.index:
    s30_sentence = (
        f'S30-only had an F1 score of {s30_f1:.1f}%, which should be interpreted relative to L30 and HLS '
        'to assess whether Sentinel-2 alone is stable enough for this CCDC setup.'
    )
else:
    s30_sentence = 'S30-only results were not available in the validation table.'

validation_setup = {
    'site': SITE_CONFIG.get('site_name', 'unknown'),
    'period': f"{globals().get('HANSEN_VALIDATION_START_YEAR', SITE_CONFIG.get('start_year', 'unknown'))}–{globals().get('HANSEN_VALIDATION_END_YEAR', SITE_CONFIG.get('end_year', 'unknown'))}",
    'map_band': globals().get('map_band', 'unknown'),
    'hansen_treecover_min': globals().get('HANSEN_VALIDATION_TREECOVER_MIN', 'unknown'),
    'hansen_scale': globals().get('HANSEN_FINE_SCALE', 'unknown'),
    'ccdc_scale': globals().get('CCDC_VALIDATION_SCALE', globals().get('map_scale', 'unknown')),
    'min_forest_frac': globals().get('HANSEN_MIN_FOREST_FRAC', 'unknown'),
    'loss_frac_threshold': globals().get('HANSEN_LOSS_FRAC_THRESHOLD', 'unknown'),
}

ccdc_extent_txt = 'not recorded'
if 'ccdc_extent' in globals():
    ccdc_extent_txt = '[' + ', '.join(f'{v:.4f}' for v in ccdc_extent) + ']'

fig4_report_path = globals().get('fig4_path', 'not available')
fig3_report_path = globals().get('fig3_path', 'not available')

report_lines = [
    f'# Hansen Zonal Validation Report: {validation_setup["site"]}',
    '',
    '## Validation Setup',
    f'- Site: `{validation_setup["site"]}`',
    f'- Period: `{validation_setup["period"]}`',
    f'- CCDC band evaluated: `{validation_setup["map_band"]}`',
    f'- Validation extent: derived from loaded CCDC GeoJSON bounds, `{ccdc_extent_txt}`',
    f'- Hansen baseline forest threshold: `treecover2000 >= {validation_setup["hansen_treecover_min"]}%`',
    f'- Hansen sampling scale: `{validation_setup["hansen_scale"]} m`',
    f'- CCDC validation grid scale: `{validation_setup["ccdc_scale"]} m`',
    f'- Eligible forest-cell threshold: `forest_frac >= {validation_setup["min_forest_frac"]}`',
    f'- Hansen loss label threshold: `loss_frac_of_forest >= {validation_setup["loss_frac_threshold"]}`',
    '',
    '## Statistical Results',
    markdown_table(metrics_pct),
    '',
    '## Main Interpretation',
    f'- {overall_sentence}',
    f'- {recall_sentence}',
    f'- {precision_sentence}',
    f'- {iou_sentence}',
    f'- {tradeoff_sentence}',
    f'- {s30_sentence}',
    '',
    '## Conclusion',
    'The validation should be interpreted as agreement with Hansen Global Forest Change, not absolute ground truth. Hansen captures stand-replacing tree-cover loss and may miss partial degradation, understory disturbance, or disturbance timing differences captured by CCDC. The key result is the sensor tradeoff: the highest-recall sensor is best for minimizing missed disturbance, while the highest-F1/IoU sensor is best aligned with Hansen in this validation setup.',
    '',
    '## Related Outputs',
    f'- CCDC disturbance map: `{fig3_report_path}`',
    f'- Hansen validation figure: `{fig4_report_path}`',
]

report_text = '\n'.join(report_lines) + '\n'
report_path = os.path.join(
    SITE_CONFIG['output_dir'],
    f"{SITE_CONFIG['site_name']}_hansen_validation_report.md",
)
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(f'Hansen validation report written → {report_path}')
print('\nPreview:')
print('\n'.join(report_lines[:35]))

---
## Summary and Export Check
This final cell prints a compact run summary so you can verify the site, observation counts, breakpoints, and saved outputs before publishing results.


In [ ]:
print('=' * 60)
print(f'Site: {SITE_CONFIG["site_name"]}')
print(f'Period: {SITE_CONFIG["start_year"]}–{SITE_CONFIG["end_year"]}')
print()
print('Observation counts:')
print(f'  L30:  {n_l30:>5d} images')
print(f'  S30:  {n_s30:>5d} images')
print(f'  HLS:  {n_hls:>5d} images  ({n_hls/max(n_l30,1):.1f}x L30-only)')
print()
print('Breakpoints detected at study pixel:')
for label, ccdc in [('L30', ccdc_l30), ('S30', ccdc_s30), ('HLS', ccdc_hls)]:
    if ccdc:
        breaks = [decimal_to_datetime(t).strftime('%Y-%m')
                  for t in (ccdc.get('tBreak') or []) if t and t > 0]
        print(f'  {label}: {breaks}')
print()
print('Saved outputs:')
output_paths = [
    ('Fig 1', globals().get('fig1_path')),
    ('Fig 2 band figures', globals().get('fig2_paths')),
    ('Snapshot figure', globals().get('SNAP_OUT')),
    ('Fig 3', globals().get('fig3_path')),
    ('Hansen preview', globals().get('hansen_viz_path')),
    ('Fig 4', globals().get('fig4_path')),
    ('Diagnostics', globals().get('diag_path')),
    ('Validation report', globals().get('report_path')),
]
for label, path in output_paths:
    if path:
        print(f'  {label}: {path}')
print('=' * 60)